In [ ]:
# NOTEBOOK NAME
# MassFeatureStatTimeChunks.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

import math
import calendar

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions/')
from CustomFunctions1 import *
from RadarPlotsCustomFunctions import *
from minisom import *

from pyproj import Geod

# KMeans clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler # normalising data for distance calculations

# for kernel density map
import cartopy.crs as ccrs
from scipy.stats import gaussian_kde


# land/ocean buffer stuff
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy.ndimage import distance_transform_edt, label

from pathlib               import Path
from matplotlib.colors     import ListedColormap, BoundaryNorm
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import glob

In [ ]:
# CHAD MAP
# OPEN OCEAN/ COASTAL OCEAN/ COASTAL LAND/ INNER LAND DEFINITION MAP

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
RadarIDno        = 22
CoastBufferKM    = 25.0        # ← updated to match SparseFXR classification
MinIslandAreaKM2 = 25.0        # ← updated to match SparseFXR classification
GEBCOpath        = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
ExampleRadarPath = '/scratch/v46/sg3241/tmp/CustomRadarGrids/QC2/Radar22/2024/20240101/22_20240101_000000_QC2.nc'
LonShift         = 0.0
# ── END USER SETTINGS ─────────────────────────────────────────────────────────


# ── STEP 1: GRAB PLOT EXTENT ──────────────────────────────────────────────────
RadarXR = xr.open_dataset(ExampleRadarPath)
LonMin  = float(RadarXR.lon.min()) + LonShift
LonMax  = float(RadarXR.lon.max()) + LonShift
LatMin  = float(RadarXR.lat.min())
LatMax  = float(RadarXR.lat.max())


# ── STEP 2: LOAD AND CLIP GEBCO ───────────────────────────────────────────────
Margin    = 0.2
GEBCO     = xr.open_dataset(GEBCOpath)
elevation = GEBCO['elevation'].sel(
    lon = slice(LonMin - Margin, LonMax + Margin),
    lat = slice(LatMin - Margin, LatMax + Margin),
)

ElevVals  = elevation.values
GEBCOlons = elevation.lon.values
GEBCOlats = elevation.lat.values


# ── STEP 3: PIXEL AREA AND SAMPLING ──────────────────────────────────────────
dLat_deg     = abs(float(GEBCOlats[1] - GEBCOlats[0]))
dLon_deg     = abs(float(GEBCOlons[1] - GEBCOlons[0]))
MidLat       = (LatMin + LatMax) / 2.0
RadiusEarth  = 6371.0
dLat_km      = dLat_deg * (RadiusEarth * 2 * np.pi / 360.0)
dLon_km      = dLon_deg * (RadiusEarth * 2 * np.pi / 360.0) * np.cos(np.deg2rad(MidLat))
PixelAreaKM2 = dLat_km * dLon_km
Sampling     = (dLat_km, dLon_km)


# ── STEP 4: LAND MASK AND CONNECTED REGION LABELLING ─────────────────────────
IsLand = ElevVals > 0.0

ConnectivityStructure  = np.ones((3, 3), dtype=int)
LandLabels, NumRegions = label(IsLand, structure=ConnectivityStructure)

RegionPixelCounts = np.bincount(LandLabels.ravel())
RegionAreaKM2     = RegionPixelCounts * PixelAreaKM2

IsLargeLand = np.zeros_like(IsLand, dtype=bool)
for RegionID in range(1, NumRegions + 1):
    if RegionAreaKM2[RegionID] >= MinIslandAreaKM2:
        IsLargeLand[LandLabels == RegionID] = True


# ── STEP 5: DISTANCE TO COASTLINE ────────────────────────────────────────────
DistLandToOcean = distance_transform_edt(IsLand,       sampling=Sampling)
DistOceanToLand = distance_transform_edt(~IsLargeLand, sampling=Sampling)
DistToCoast     = DistLandToOcean + DistOceanToLand


# ── STEP 6: CLASSIFY ─────────────────────────────────────────────────────────
Classification = np.full(ElevVals.shape, -1, dtype=np.int8)
Classification[ IsLand & (DistToCoast >  CoastBufferKM)] = 0
Classification[ IsLand & (DistToCoast <= CoastBufferKM)] = 1
Classification[~IsLand & (DistToCoast <= CoastBufferKM)] = 2
Classification[~IsLand & (DistToCoast >  CoastBufferKM)] = 3

SmallIslandMask            = IsLand & ~IsLargeLand
Classification[SmallIslandMask] = 1


# ── STEP 7: PLOT ──────────────────────────────────────────────────────────────
ClassColours = [
    '#4a7c4e',   # 0: inner land
    '#8fbf6a',   # 1: coastal land
    '#5aafd4',   # 2: coastal ocean
    '#1a5f8a',   # 3: open ocean
]
CMap = mcolors.ListedColormap(ClassColours)
Norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], CMap.N)

fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#0a0a0a')

LonGrid, LatGrid = np.meshgrid(GEBCOlons, GEBCOlats)

ax.pcolormesh(
    LonGrid, LatGrid, Classification,
    cmap=CMap, norm=Norm,
    transform=ccrs.PlateCarree(),
    zorder=5,
)

# ── GEBCO-DERIVED COASTLINE ───────────────────────────────────────────────────
ax.contour(
    GEBCOlons,
    GEBCOlats,
    ElevVals,
    levels    = [0.0],
    colors    = ['white'],
    linewidths= [0.5],
    transform = ccrs.PlateCarree(),
    zorder    = 10,
)

# ── RADAR LOCATION STAR ───────────────────────────────────────────────────────
ax.plot(float(RadarXR.radar_longitude[0]) + LonShift, float(RadarXR.radar_latitude[0]),
        marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=30)
ax.plot(float(RadarXR.radar_longitude[0]) + LonShift, float(RadarXR.radar_latitude[0]),
        marker='*', color='white', markersize=4, transform=ccrs.PlateCarree(), zorder=31)

# ── LAT/LON GRID LINES ────────────────────────────────────────────────────────
ThinLineThickness   = 0.2
MediumLineThickness = 0.3
ThickLineThickness  = 0.6
ThinLineFrequency   = 0.1
MediumLineFrequency = 0.5
ThickLineFrequency  = 1.0
StandOutColour      = 'white'

AddGridlines2(ax, ThinLineThickness, MediumLineThickness, ThickLineThickness,
                  ThinLineFrequency,  MediumLineFrequency,  ThickLineFrequency, StandOutColour)

# ── LEGEND ────────────────────────────────────────────────────────────────────
patches = [
    mpatches.Patch(facecolor=ClassColours[0], label='Inner Land'),
    mpatches.Patch(facecolor=ClassColours[1], label='Coastal Land'),
    mpatches.Patch(facecolor=ClassColours[2], label='Coastal Ocean'),
    mpatches.Patch(facecolor=ClassColours[3], label='Open Ocean'),
]
legend = ax.legend(
    handles=patches, loc='upper right',
    facecolor='#0a0a0a', labelcolor='white', edgecolor='white',
    fontsize=9, borderpad=0.6, labelspacing=0.35,
)
legend.get_frame().set_alpha(1.0)
legend.set_zorder(50)

ax.set_extent([LonMin, LonMax, LatMin, LatMax], crs=ccrs.PlateCarree())
plt.title(
    f'Mackay Radar Domain Region Classifications\n'
    f'{CoastBufferKM:.0f} km Coastal Buffer  (Min Island Area {MinIslandAreaKM2:.0f} km²)',
    color='white', pad=10,
)
plt.tight_layout()

# # ── SAVE ──────────────────────────────────────────────────────────────────────
# SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/'
# SaveFile   = f'Radar{RadarIDno}DomainRegionClassifications'
# SavePath   = SaveFolder + SaveFile

# if not Path(SaveFolder).exists():
#     Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='k', dpi=300)


In [ ]:
# CHAD FUNCTION
# ADDS COASTAL BUFFER CATEGORIES (OPEN OCEAN/ COASTAL OCEAN) TO EACH TRACK AND FEATUREFRAME

def AddCoastalClassificationToSparseFXR(
    SparseFXR,
    RadarIDno,
    QualityControlOption,
    GEBCOpath,
    CoastBufferKM    = 25.0,
    MinIslandAreaKM2 = 25.0,
):
    """
    Computes a 4-class coastal/land classification for every feature frame in
    SparseFXR and adds three new variables:

        FeatureFrameLandClassification  (sparse_index)
            The coastal class of each feature at each frame.

        BirthLandGroupClassification    (tracks)
            The coastal class of each track at its birth frame.

        DeathLandGroupClassification    (tracks)
            The coastal class of each track at its death frame.

    Classification integer codes
    ────────────────────────────
        0 : Inner Land    — elevation > 0 m, > CoastBufferKM from coast
        1 : Coastal Land  — elevation > 0 m, ≤ CoastBufferKM from coast
        2 : Coastal Ocean — elevation ≤ 0 m, ≤ CoastBufferKM from large land
        3 : Open Ocean    — elevation ≤ 0 m, > CoastBufferKM from large land

    Small islands (area < MinIslandAreaKM2) are always classified as
    Coastal Land (1) but do not generate a Coastal Ocean buffer around them.

    Args:
        SparseFXR            : xarray.Dataset — the sparse feature dataset
        RadarIDno            : int   — radar ID number (e.g. 22)
        QualityControlOption : int or str — QC option used (e.g. 2)
        GEBCOpath            : str   — path to the GEBCO NetCDF elevation file
        CoastBufferKM        : float — buffer distance defining "coastal" [km]
        MinIslandAreaKM2     : float — minimum island area to generate ocean
                                       buffer [km²]

    Returns:
        SparseFXR : xarray.Dataset with three new variables added in-place
    """

    from scipy.ndimage import distance_transform_edt, label as ndimage_label
    import xarray as xr
    import numpy as np

    ClassLegend = (
        '0=Inner Land (elev>0m, >CoastBufferKM from coast); '
        '1=Coastal Land (elev>0m, <=CoastBufferKM from coast, '
        'or small island); '
        '2=Coastal Ocean (elev<=0m, <=CoastBufferKM from large land); '
        '3=Open Ocean (elev<=0m, >CoastBufferKM from large land)'
    )

    # ── STEP 1: DERIVE DATE AND BUILD SAMPLE RADAR PATH ──────────────────────
    # startdate attribute is formatted as 'YYYYMMDD.HHMMSS'
    StartDateStr = SparseFXR.attrs['startdate']   # e.g. '20240201.000000'
    DateStr      = StartDateStr.split('.')[0]      # e.g. '20240201'
    YearStr      = DateStr[0:4]                    # e.g. '2024'

    SampleRadarPath = (
        f'/scratch/v46/sg3241/tmp/CustomRadarGrids/'
        f'QC{QualityControlOption}/'
        f'Radar{RadarIDno}/'
        f'{YearStr}/{DateStr}/'
        f'{RadarIDno}_{DateStr}_000000_QC{QualityControlOption}.nc'
    )

    print(f'Loading sample radar file for domain extent: {SampleRadarPath}')
    RadarXR = xr.open_dataset(SampleRadarPath)
    LonMin  = float(RadarXR.lon.min())
    LonMax  = float(RadarXR.lon.max())
    LatMin  = float(RadarXR.lat.min())
    LatMax  = float(RadarXR.lat.max())
    print(f'Domain extent — Lon: [{LonMin:.2f}, {LonMax:.2f}]  '
          f'Lat: [{LatMin:.2f}, {LatMax:.2f}]')


    # ── STEP 2: LOAD AND CLIP GEBCO ──────────────────────────────────────────
    Margin    = 0.2
    GEBCO     = xr.open_dataset(GEBCOpath)
    elevation = GEBCO['elevation'].sel(
        lon = slice(LonMin - Margin, LonMax + Margin),
        lat = slice(LatMin - Margin, LatMax + Margin),
    )

    ElevVals  = elevation.values
    GEBCOlons = elevation.lon.values
    GEBCOlats = elevation.lat.values


    # ── STEP 3: PIXEL AREA AND SAMPLING ──────────────────────────────────────
    dLat_deg     = abs(float(GEBCOlats[1] - GEBCOlats[0]))
    dLon_deg     = abs(float(GEBCOlons[1] - GEBCOlons[0]))
    MidLat       = (LatMin + LatMax) / 2.0
    RadiusEarth  = 6371.0
    dLat_km      = dLat_deg * (RadiusEarth * 2 * np.pi / 360.0)
    dLon_km      = dLon_deg * (RadiusEarth * 2 * np.pi / 360.0) * np.cos(np.deg2rad(MidLat))
    PixelAreaKM2 = dLat_km * dLon_km
    Sampling     = (dLat_km, dLon_km)


    # ── STEP 4: LAND MASK AND CONNECTED REGION LABELLING ─────────────────────
    IsLand = ElevVals > 0.0

    ConnectivityStructure  = np.ones((3, 3), dtype=int)
    LandLabels, NumRegions = ndimage_label(IsLand, structure=ConnectivityStructure)
    print(f'Found {NumRegions} connected land regions in clipped GEBCO domain')

    RegionPixelCounts = np.bincount(LandLabels.ravel())
    RegionAreaKM2     = RegionPixelCounts * PixelAreaKM2

    IsLargeLand = np.zeros_like(IsLand, dtype=bool)
    for RegionID in range(1, NumRegions + 1):
        if RegionAreaKM2[RegionID] >= MinIslandAreaKM2:
            IsLargeLand[LandLabels == RegionID] = True


    # ── STEP 5: DISTANCE TO COASTLINE ────────────────────────────────────────
    print('Computing distance transforms — this may take a moment...')
    DistLandToOcean = distance_transform_edt(IsLand,       sampling=Sampling)
    DistOceanToLand = distance_transform_edt(~IsLargeLand, sampling=Sampling)
    DistToCoast     = DistLandToOcean + DistOceanToLand


    # ── STEP 6: BUILD CLASSIFICATION GRID ────────────────────────────────────
    ClassGrid = np.full(ElevVals.shape, -1, dtype=np.int8)
    ClassGrid[ IsLand & (DistToCoast >  CoastBufferKM)] = 0
    ClassGrid[ IsLand & (DistToCoast <= CoastBufferKM)] = 1
    ClassGrid[~IsLand & (DistToCoast <= CoastBufferKM)] = 2
    ClassGrid[~IsLand & (DistToCoast >  CoastBufferKM)] = 3

    # Small islands always coastal land regardless of buffer distance
    SmallIslandMask          = IsLand & ~IsLargeLand
    ClassGrid[SmallIslandMask] = 1


    # ── STEP 7: NEAREST-NEIGHBOUR LOOKUP FOR EVERY SPARSE FEATURE FRAME ──────
    # Convert feature lon/lat to the nearest GEBCO pixel index and read off
    # the class value — no interpolation, just an index lookup.
    print('Assigning classification to each sparse feature frame...')

    FeatureLons = SparseFXR['meanlon'].values   # (sparse_index,)
    FeatureLats = SparseFXR['meanlat'].values   # (sparse_index,)

    # Find the nearest GEBCO grid index for each feature centre
    # np.searchsorted is fast for sorted 1-D arrays
    LonIndices = np.searchsorted(GEBCOlons, FeatureLons) 
    LatIndices = np.searchsorted(GEBCOlats, FeatureLats)

    # searchsorted returns the insertion point — we want the nearest neighbour
    # so we check both the index and index-1 and pick whichever is closer
    LonIndices = np.clip(LonIndices, 0, len(GEBCOlons) - 1)
    LatIndices = np.clip(LatIndices, 0, len(GEBCOlats) - 1)

    # For each feature, compare distance to the pixel just below and just above
    # and nudge the index down by one where the lower pixel is actually closer
    LonIndices_m1 = np.clip(LonIndices - 1, 0, len(GEBCOlons) - 1)
    LatIndices_m1 = np.clip(LatIndices - 1, 0, len(GEBCOlats) - 1)

    LonCloser = np.abs(FeatureLons - GEBCOlons[LonIndices_m1]) < \
                np.abs(FeatureLons - GEBCOlons[LonIndices])
    LatCloser = np.abs(FeatureLats - GEBCOlats[LatIndices_m1]) < \
                np.abs(FeatureLats - GEBCOlats[LatIndices])

    LonIndices[LonCloser] = LonIndices_m1[LonCloser]
    LatIndices[LatCloser] = LatIndices_m1[LatCloser]

    # Read the class value at each (lat, lon) index pair
    # ClassGrid has shape (lat, lon) so index as [lat_idx, lon_idx]
    FeatureFrameClass = ClassGrid[LatIndices, LonIndices]   # (sparse_index,)


    # ── STEP 8: DERIVE BIRTH AND DEATH CLASSIFICATIONS (size: tracks) ─────────
    print('Deriving birth and death classifications...')

    FeatureFrames  = SparseFXR['times_indices'].values    # (sparse_index,)
    TrackDurations = SparseFXR['track_duration'].values   # (tracks,)
    FeatureIndices = SparseFXR['tracks_indices'].values   # (sparse_index,)

    BirthMask = (FeatureFrames == 0)
    DeathMask = (FeatureFrames == TrackDurations[FeatureIndices.astype(int)] - 1)

    # Extract the class at birth and death frames — one value per track
    BirthClass = FeatureFrameClass[BirthMask]   # (tracks,)
    DeathClass  = FeatureFrameClass[DeathMask]   # (tracks,)


    # ── STEP 9: ADD VARIABLES BACK TO SparseFXR ───────────────────────────────
    print('Adding new variables to SparseFXR...')

    SharedAttrs = {
        'units'            : '1',
        'flag_values'      : '0, 1, 2, 3',
        'flag_meanings'    : ClassLegend,
        'CoastBufferKM'    : CoastBufferKM,
        'MinIslandAreaKM2' : MinIslandAreaKM2,
    }

    SparseFXR['FeatureFrameLandClassification'] = xr.Variable(
        dims  = 'sparse_index',
        data  = FeatureFrameClass,
        attrs = {
            'long_name' : 'Coastal/land classification at each feature frame',
            **SharedAttrs,
        }
    )

    SparseFXR['BirthLandGroupClassification'] = xr.Variable(
        dims  = 'tracks',
        data  = BirthClass,
        attrs = {
            'long_name' : 'Coastal/land classification at track birth location',
            **SharedAttrs,
        }
    )

    SparseFXR['DeathLandGroupClassification'] = xr.Variable(
        dims  = 'tracks',
        data  = DeathClass,
        attrs = {
            'long_name' : 'Coastal/land classification at track death location',
            **SharedAttrs,
        }
    )

    print('Done. Three new variables added to SparseFXR:')
    print('  FeatureFrameLandClassification  (sparse_index)')
    print('  BirthLandGroupClassification    (tracks)')
    print('  DeathLandGroupClassification    (tracks)')

    return SparseFXR


In [ ]:
# LOAD IN MASS FEATURE STATS NETCDF (IN SPARSE FORMAT)

# USER CHOICE SECTION
StartFileDateStr = '20240101'
EndFileDateStr = '20240229'
RadarIDno = 22
QualityControlOption = 2
TrackingVersionNumber = 5

# THIS NEEDS TO BE UPDATED TO FIND WHERE NEW SPARSE STATS ARE STORED 
# THIS NEEDS TO BE UPDATED TO FIND WHERE NEW SPARSE STATS ARE STORED 
# THIS NEEDS TO BE UPDATED TO FIND WHERE NEW SPARSE STATS ARE STORED 
# THIS NEEDS TO BE UPDATED TO FIND WHERE NEW SPARSE STATS ARE STORED 
# THIS NEEDS TO BE UPDATED TO FIND WHERE NEW SPARSE STATS ARE STORED 
# THIS NEEDS TO BE UPDATED TO FIND WHERE NEW SPARSE STATS ARE STORED 

# load in the sparse stats
SparseStoragePath  = '/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/Stats/Radar22/QC2/V5/Period_20240201_20240229/trackstats_sparse_20240201.000000_20240229.235500.nc'

               # home/tmp/PyFLEXTRKRnetCDFs/V5/QC2/Radar22/Period_20240101_20240229/Stats/tracknumbers_20240101.000000_20240229.235500.nc

# SparseStoragePath = (f'/scratch/v46/sg3241/tmp/PyFLEXTRKRnetCDFs/V{TrackingVersionNumber}/QC{QualityControlOption}/Radar{RadarIDno}/'
#                      f'Period_{StartFileDateStr}_{EndFileDateStr}/Stats/tracknumbers_{StartFileDateStr}.000000_{EndFileDateStr}.235500.nc')

SparseFXR = xr.open_dataset(SparseStoragePath)



# add to the sparse data set a time variable that lists the time rounded down to 5-mins AS IN THE NAME OF THE RADAR SCAN
SparseFXR['base_time_fivemin'] = xr.DataArray(pd.DatetimeIndex(SparseFXR['base_time'].values).floor('5min').to_numpy(), dims='sparse_index')
SparseFXR['base_time_fivemin'].attrs['units']     = SparseFXR['base_time'].attrs.get('units',     'unknown')
SparseFXR['base_time_fivemin'].attrs['long_name'] = SparseFXR['base_time'].attrs.get('long_name', 'base_time') + ' rounded down to lower 5-minutes'

# add to the sparse data set an index time variable that lists the number of 5-mins scans into the period you are
t0 = SparseFXR['base_time_fivemin'].values[0]
SparseFXR['base_time_index'] = xr.DataArray( ((SparseFXR['base_time_fivemin'].values - t0) / np.timedelta64(5, 'm')).astype(int), dims='sparse_index')
SparseFXR['base_time_index'].attrs['units']     = 'sets of 5 minutes'
SparseFXR['base_time_index'].attrs['long_name'] = 'number of 5-minute frames into the period'

In [ ]:
SparseFXR = AddCoastalClassificationToSparseFXR(
    SparseFXR            = SparseFXR,
    RadarIDno            = 22,
    QualityControlOption = 2,
    GEBCOpath            = '/home/563/sg3241/QueenslandElevationGEBCO.nc',
    CoastBufferKM        = 25.0,  # distance from the coast considered "coastal" for both land and ocean
    MinIslandAreaKM2     = 10.0,  # the smallest islands that will count towards the "coastal ocean" buffer
)

In [ ]:
def AddSparseIndexedVars(ds):
    """
    For every variable with only the (tracks) dimension, create a new
    variable with the suffix '_sparse_indexed' and dimension (sparse_index),
    by mapping through tracks_indices.

    Skips creation if the '_sparse_indexed' variable already exists in the dataset.

    Parameters
    ----------
    ds : xr.Dataset

    Returns
    -------
    xr.Dataset with additional '_sparse_indexed' variables
    """
    ds_new        = ds.copy()
    track_indices = ds['tracks_indices'].values.astype(int)

    for var in ds.data_vars:
        if ds[var].dims == ('tracks',):
            new_var = f'{var}_sparse_indexed'

            # skip if this variable has already been created
            if new_var in ds_new.data_vars:
                continue

            ds_new[new_var] = xr.DataArray(ds[var].values[track_indices], dims='sparse_index')
            ds_new[new_var].attrs['units']     = ds[var].attrs.get('units',     'unknown')
            ds_new[new_var].attrs['long_name'] = ds[var].attrs.get('long_name', var) + ' indexed from feature-frame time'

    return ds_new


In [ ]:
# add the 'sparse-indexed' variable for convenience later
SparseFXR = AddSparseIndexedVars(SparseFXR)


In [ ]:
# add propogation vector data to the feature stats

# retrive an array for the amount of time that passes between each feature time frame (might not always be exactly 5 min)
TimeTravels = np.concatenate([[np.timedelta64('NaT')], np.diff(BaseTimesFiveMin)]) # starts as a time delta array
TimeTravels = TimeTravels / np.timedelta64(1, 's')                                  # change the unit from ns to s

# remove difference data for the first frame of each feature's life since it hasn't gone anywhere yet
TimeTravels[BirthMask == 1] = np.nan # now a float array (number of seconds)


# retrieve the coordinates of feature locations at each feature time frame
FeatureLats = SparseFXR['meanlat'].values
FeatureLons = SparseFXR['meanlon'].values

# set up arrays for where each feature begins and ends between each time frame
StartLats = FeatureLats[:-1]
EndLats   = FeatureLats[1:]

StartLons = FeatureLons[:-1]
EndLons   = FeatureLons[1:]

# calculate cartesian distances and directions travelled given lats and lons  --- from a geodesic calculation ---
GEOD = Geod(ellps='WGS84')
Azimuths, BackAzimuths, DistTravels = GEOD.inv(StartLons, StartLats, EndLons , EndLats)

# add a NaN to the beginning of each difference array to keep it NumFeatureFrames long
Azimuths = np.concatenate([[np.nan], Azimuths]) 
BackAzimuths = np.concatenate([[np.nan], BackAzimuths]) 
DistTravels = np.concatenate([[np.nan], DistTravels]) 

# remove difference data for the first frame of each feature's life since it hasn't gone anywhere yet
Azimuths[BirthMask == 1] = np.nan
BackAzimuths[BirthMask == 1] = np.nan
DistTravels[BirthMask == 1] = np.nan

# convert travel direction to wind direction
PropDirections = Azimuths + 180

# calculate travel distances in purely cartesian directions [m]
Xtravels = DistTravels * np.cos(90 - Azimuths)
Ytravels = DistTravels * np.sin(90 - Azimuths)


# --- Compute speeds [m/s] and direction ---
PropDirections = Azimuths + 180
PropSpeeds     = DistTravels / TimeTravels
Uspeeds        = Xtravels / TimeTravels
Vspeeds        = Ytravels / TimeTravels

SparseFXR['PropDirections'] = xr.DataArray(PropDirections, dims=['sparse_index'])
SparseFXR['PropDirections'].attrs['units']     = 'degrees'
SparseFXR['PropDirections'].attrs['long_name'] = 'Propagation Direction (azimuthal)'

SparseFXR['PropSpeeds'] = xr.DataArray(PropSpeeds, dims=['sparse_index'])
SparseFXR['PropSpeeds'].attrs['units']     = 'm/s'
SparseFXR['PropSpeeds'].attrs['long_name'] = 'Propagation Speed'

SparseFXR['Uspeeds'] = xr.DataArray(Uspeeds, dims=['sparse_index'])
SparseFXR['Uspeeds'].attrs['units']     = 'm/s'
SparseFXR['Uspeeds'].attrs['long_name'] = 'East-West Speed Component'

SparseFXR['Vspeeds'] = xr.DataArray(Vspeeds, dims=['sparse_index'])
SparseFXR['Vspeeds'].attrs['units']     = 'm/s'
SparseFXR['Vspeeds'].attrs['long_name'] = 'North-South Speed Component'

In [ ]:
SparseFXR

In [ ]:
occurences(SparseFXR['FeatureFrameLandClassification'].values)

In [ ]:
# Create All of the Masks you need

NumFeatureFrames = np.size(SparseFXR['sparse_index'].values)
NumFeatures = np.size(SparseFXR['tracks'].values)

RangeStartHour = 4
RangeEndHour = 6

MinDuration = 12 # [minutes]

# minimum number of frames required to meet time requirement
MinFrames = int(np.ceil(MinDuration/5) + 1)

# turn some variables into numpy arrays
BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)
# TopoAltitudes    = SparseFXR['feature_centre_topo_alt'].values
FeatureFrames    = SparseFXR['times_indices'].values
FeatureIndices   = SparseFXR['tracks_indices'].values
TrackDurations   = SparseFXR['track_duration_sparse_indexed'].values

# # mask to choose only those feature-frames over land or over the ocean
# FeatureFrameLandMask  = (TopoAltitudes > 0)
# FeatureFrameOceanMask = ~FeatureFrameLandMask

# FeatureLandMask = np.full(NumFeatures, np.nan)

# masks for those sparse indicies which reperesent when a feature was born or died
BirthMask = (FeatureFrames == 0)
DeathMask = (FeatureFrames == TrackDurations-1)

# # create masks SIZE NUM_FEATURES which say if a feature was born/died over the ocean/land
# BirthLandMask  = FeatureFrameLandMask[BirthMask]
# BirthOceanMask = ~BirthLandMask
# DeathLandMask  = FeatureFrameLandMask[DeathMask]
# DeathOceanMask = ~DeathLandMask

# # create masks SIZE NUM_FEATURES_FRAMES which say if a feature was born/died over the ocean/land
# SparseTrackBirthLandMask  = BirthLandMask[FeatureIndices.astype(int)]
# SparseTrackBirthOceanMask = BirthOceanMask[FeatureIndices.astype(int)]
# SparseTrackDeathLandMask  = DeathLandMask[FeatureIndices.astype(int)]
# SparseTrackDeathOceanMask = DeathOceanMask[FeatureIndices.astype(int)]

# mask to only choose those features in this time chunk
TimeMask  = (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute >= RangeStartHour*60) & (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute < RangeEndHour*60)

# mask to only include those features that last a certain amount of time
DurationMask = (TrackDurations >= MinFrames)

In [ ]:
# BIRTH AND DEATH LOCATION MASKS

BirthClasses = SparseFXR['BirthLandGroupClassification_sparse_indexed'].values
DeathClasses = SparseFXR['DeathLandGroupClassification_sparse_indexed'].values

OceanBirthLandDeathMask = ( (BirthClasses == 3) & (DeathClasses == 0) )
LandBirthOceanDeathMask = ( (BirthClasses < 1.5) & (DeathClasses == 3) )

FeatureAreas = SparseFXR['area'].values

OnshoreAreas   = FeatureAreas[OceanBirthLandDeathMask * DurationMask]
OffshoreAreas  = FeatureAreas[LandBirthOceanDeathMask * DurationMask]

OnshoreFeatureFrames    = FeatureFrames[OceanBirthLandDeathMask * DurationMask]
OnshoreFeatureIndices   = FeatureIndices[OceanBirthLandDeathMask * DurationMask]

OffshoreFeatureFrames   = FeatureFrames[LandBirthOceanDeathMask * DurationMask]
OffshoreFeatureIndices   = FeatureIndices[LandBirthOceanDeathMask * DurationMask]

In [ ]:
print(OnshoreFeatureIndices)

In [ ]:
np.mean(OffshoreAreas)

In [ ]:
plt.hist(LandwardAreas,100)

In [ ]:
# CHAD FUNCTION
# CREATES ARRAYS STORING DATA FOR EACH TRACK AT EACH FRAME UP TO THE MINIMUM NUMBER OF FRAMES

def BuildTrackFrameGrid(
    SparseFXR,
    VariableName,
    MinDurationMinutes,
    BirthClasses,
    DeathClasses,
):
    """
    Filters tracks by duration and birth/death geographic classification,
    then builds a 2D grid of shape (NumTracks, MinFrames) from a sparse
    variable in SparseFXR.

    Only the first MinFrames frames of each qualifying track are used.
    Frames beyond MinFrames are discarded. Any cells with no data are
    filled with np.nan.

    Args:
        SparseFXR          : xarray.Dataset — the sparse feature dataset
        VariableName       : str            — name of the variable in SparseFXR
                                              to extract (dim: sparse_index)
        MinDurationMinutes : float          — minimum track duration in minutes
                                              (tracks shorter than this are excluded)
        BirthClasses       : list of int    — allowed geographic class codes for
                                              the birth location (e.g. [2, 3] for
                                              any ocean birth)
        DeathClasses       : list of int    — allowed geographic class codes for
                                              the death location (e.g. [0, 1] for
                                              any land death)

    Returns:
        Grid               : np.ndarray, shape (NumTracks, MinFrames), dtype float
                             rows re-indexed 0..NumTracks-1
        TrackIDs           : np.ndarray, shape (NumTracks,)
                             original track numbers corresponding to each row
        MinFrames          : int
                             number of frames columns in the grid
    """

    # ── DERIVE MinFrames FROM MinDurationMinutes ──────────────────────────────
    MinFrames = int(np.ceil(MinDurationMinutes / 5) + 1)

    # ── PULL OUT THE ARRAYS WE NEED ───────────────────────────────────────────
    TrackDurations  = SparseFXR['track_duration'].values          # (tracks,)
    FeatureFrames   = SparseFXR['times_indices'].values           # (sparse_index,)
    FeatureIndices  = SparseFXR['tracks_indices'].values          # (sparse_index,)
    BirthClassArr   = SparseFXR['BirthLandGroupClassification_sparse_indexed'].values   # (sparse_index,)
    DeathClassArr   = SparseFXR['DeathLandGroupClassification_sparse_indexed'].values   # (sparse_index,)
    SparseValues    = SparseFXR[VariableName].values.astype(float)                      # (sparse_index,)

    # ── DURATION MASK (sparse_index) ─────────────────────────────────────────
    # TrackDurations is size (tracks,) so index it by FeatureIndices to get
    # a per-sparse-frame duration value, then apply the minimum frame threshold
    DurationMask = (TrackDurations[FeatureIndices.astype(int)] >= MinFrames)

    # ── BIRTH / DEATH CLASS MASKS (sparse_index) ──────────────────────────────
    # Build a boolean mask for each allowed birth class and OR them together
    BirthMask = np.zeros(len(FeatureFrames), dtype=bool)
    for ClassCode in BirthClasses:
        BirthMask |= (BirthClassArr == ClassCode)

    DeathMask = np.zeros(len(FeatureFrames), dtype=bool)
    for ClassCode in DeathClasses:
        DeathMask |= (DeathClassArr == ClassCode)

    # ── COMBINED MASK ─────────────────────────────────────────────────────────
    CombinedMask = DurationMask & BirthMask & DeathMask

    # ── APPLY MASK TO GET FILTERED SPARSE ARRAYS ─────────────────────────────
    FilteredFrames  = FeatureFrames[CombinedMask].astype(int)
    FilteredIndices = FeatureIndices[CombinedMask].astype(int)
    FilteredValues  = SparseValues[CombinedMask]

    # ── BUILD RE-INDEXED TRACK ID MAPPING ────────────────────────────────────
    TrackIDs  = np.unique(FilteredIndices)
    NumTracks = len(TrackIDs)

    if NumTracks == 0:
        print('Warning: no tracks passed the filters — returning empty grid')
        return np.full((0, MinFrames), np.nan, dtype=float), TrackIDs, MinFrames

    TrackIDtoRow = np.full(int(TrackIDs.max()) + 1, -1, dtype=int)
    TrackIDtoRow[TrackIDs] = np.arange(NumTracks)

    # ── INITIALISE OUTPUT GRID ────────────────────────────────────────────────
    Grid = np.full((NumTracks, MinFrames), np.nan, dtype=float)

    # ── ONLY KEEP FRAMES WITHIN [0, MinFrames) ────────────────────────────────
    FrameRangeMask  = FilteredFrames < MinFrames
    ValidFrames     = FilteredFrames[FrameRangeMask]
    ValidIndices    = FilteredIndices[FrameRangeMask]
    ValidValues     = FilteredValues[FrameRangeMask]

    # ── FILL THE GRID ─────────────────────────────────────────────────────────
    RowIndices = TrackIDtoRow[ValidIndices]
    Grid[RowIndices, ValidFrames] = ValidValues

    print(f'Tracks passing filters: {NumTracks}')
    print(f'Grid shape: {Grid.shape}')

    return Grid, TrackIDs, MinFrames


In [ ]:
# CHAD EXECUTION
# CREATES ARRAYS STORING DATA FOR EACH TRACK AT EACH FRAME UP TO THE MINIMUM NUMBER OF FRAMES

OnshoreGrid, OnshoreTrackIDs, MinFrames = BuildTrackFrameGrid(
    SparseFXR          = SparseFXR,
    VariableName       = 'area',
    MinDurationMinutes = 60,
    BirthClasses       = [3], # Onshore: born over open ocean (3), died over inner land (0)
    DeathClasses       = [0],
)


OffshoreGrid, OffshoreTrackIDs, MinFrames = BuildTrackFrameGrid(
    SparseFXR          = SparseFXR,
    VariableName       = 'area',
    MinDurationMinutes = 60,
    BirthClasses       = [0, 1], # Offshore: born over any land (0 or 1), died over open ocean (3)
    DeathClasses       = [3],
)

In [ ]:
# CHAD PLOT
# FEATURE LIFETIME TRAJECTORIES

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
VariableName       = 'max_dbz'
MinDurationMinutes = 120

IncludeOceanOnly  = True
IncludeOnshore    = False
IncludeLandOnly   = True
IncludeOffshore   = False
# ── END USER SETTINGS ─────────────────────────────────────────────────────────


# ── METADATA ──────────────────────────────────────────────────────────────────
VarAttrs = SparseFXR[VariableName].attrs
VarUnits = VarAttrs.get('units', '')
VarLong  = VarAttrs.get('long_name', VariableName)


# ── PERCENTILE HELPER ─────────────────────────────────────────────────────────
def ComputePercentiles(Grid):
    P25 = np.full(Grid.shape[1], np.nan)
    P50 = np.full(Grid.shape[1], np.nan)
    P75 = np.full(Grid.shape[1], np.nan)
    for i in range(Grid.shape[1]):
        Valid = Grid[:, i][~np.isnan(Grid[:, i])]
        if len(Valid) > 0:
            P25[i] = np.percentile(Valid, 25)
            P50[i] = np.percentile(Valid, 50)
            P75[i] = np.percentile(Valid, 75)
    return P25, P50, P75


# ── FIXED CATEGORY DEFINITIONS ────────────────────────────────────────────────
AllCategories = [
    {
        'Include'      : IncludeOceanOnly,
        'BirthClasses' : [3],
        'DeathClasses' : [3],
        'Colour'       : [0.05, 0.20, 0.90],
        'Label'        : 'Ocean Only',
    },
    {
        'Include'      : IncludeOnshore,
        'BirthClasses' : [3],
        'DeathClasses' : [0],
        'Colour'       : [0.00, 0.80, 0.90],
        'Label'        : 'Onshore — Ocean → Land',
    },
    {
        'Include'      : IncludeLandOnly,
        'BirthClasses' : [0],
        'DeathClasses' : [0],
        'Colour'       : [0.00, 0.45, 0.10],
        'Label'        : 'Land Only',
    },
    {
        'Include'      : IncludeOffshore,
        'BirthClasses' : [0, 1],
        'DeathClasses' : [3],
        'Colour'       : [0.40, 0.90, 0.10],
        'Label'        : 'Offshore — Land → Ocean',
    },
]


# ── BUILD GRIDS AND COMPUTE PERCENTILES ───────────────────────────────────────
MinFrames    = None
PlotDatasets = []

for Cat in AllCategories:

    if not Cat['Include']:
        continue

    Grid, TrackIDs, NFrames = BuildTrackFrameGrid(
        SparseFXR          = SparseFXR,
        VariableName       = VariableName,
        MinDurationMinutes = MinDurationMinutes,
        BirthClasses       = Cat['BirthClasses'],
        DeathClasses       = Cat['DeathClasses'],
    )

    if MinFrames is None:
        MinFrames = NFrames

    P25, P50, P75 = ComputePercentiles(Grid)

    PlotDatasets.append({
        'P25'    : P25,
        'P50'    : P50,
        'P75'    : P75,
        'Colour' : Cat['Colour'],
        'Label'  : f"{Cat['Label']} (n={len(TrackIDs)})",
    })

TimeMinutes = np.arange(MinFrames) * 5


# ── PLOT HELPER ───────────────────────────────────────────────────────────────
def PlotPercentiles(ax, Times, P25, P50, P75, Colour, Label):
    Mask = ~np.isnan(P50)
    T    = Times[Mask]
    ax.fill_between(T, P25[Mask], P75[Mask], color=Colour, alpha=0.12, linewidth=0)
    ax.plot(T, P25[Mask], color=Colour, linewidth=0.8, alpha=0.5, linestyle='-')
    ax.plot(T, P75[Mask], color=Colour, linewidth=0.8, alpha=0.5, linestyle='-')
    ax.plot(T, P50[Mask], color=Colour, linewidth=2.5, alpha=1.0, linestyle='-', label=Label)


# ── FIGURE ────────────────────────────────────────────────────────────────────
fig, ax_lines = plt.subplots(figsize=(10, 5))
ax_lines.set_facecolor('black')
fig.patch.set_facecolor('black')

for D in PlotDatasets:
    PlotPercentiles(ax_lines, TimeMinutes, D['P25'], D['P50'], D['P75'],
                    D['Colour'], D['Label'])

# ── FORMATTING ────────────────────────────────────────────────────────────────
ax_lines.set_xlabel("Time Since Feature Birth [minutes]", color='white', fontsize=13)
ax_lines.set_ylabel(f'{VarLong} [{VarUnits}]', color='white', fontsize=13)
ax_lines.set_title(
    f'{VarLong} Percentiles Throughout Feature Lifetime\n'
    f'Tracks lasting ≥ {MinDurationMinutes} min',
    color='white', fontsize=13,
)
ax_lines.tick_params(colors='white', which='both', labelsize=11)
ax_lines.grid(which='major', color='white', linewidth=0.8, alpha=0.5)
ax_lines.grid(which='minor', color='white', linewidth=0.3, alpha=0.3)
ax_lines.minorticks_on()
ax_lines.legend(facecolor='black', edgecolor='white', labelcolor='white', fontsize=11)
for Spine in ax_lines.spines.values():
    Spine.set_edgecolor('white')

ax_lines.set_xlim(0, (MinFrames - 1) * 5)

plt.tight_layout()
plt.show()


In [ ]:
# # CHAD METHOD
# # ADD TOPOGRAPHIC ELEVATION FROM UNDER EACH FEATURE

# # --- Load the GEBCO DEM (same file you already use) ---
# ElevationModelPath = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
# DEMdata = xr.open_dataset(ElevationModelPath)
# DEMelev = DEMdata['elevation']  # shape: (lat, lon)

# # --- Extract feature centre coordinates ---
# # Both have dimension 'sparse_index'
# FeatureLons = xr.DataArray(SparseFXR['meanlon'].values, dims='sparse_index')
# FeatureLats = xr.DataArray(SparseFXR['meanlat'].values, dims='sparse_index')

# # --- Interpolate DEM at each feature centre location ---
# # xarray's interp will vectorise over 'sparse_index' automatically
# # because FeatureLons and FeatureLats share the same dimension name
# TopoAlt_interp = DEMelev.interp(
#     lon = FeatureLons,
#     lat = FeatureLats,
#     method = 'linear',        # bilinear interpolation across the DEM grid
# )

# # --- Convert to a plain numpy array [metres] ---
# # Set ocean/below-sea-level values to 0.0 (features over water get 0 m altitude)
# TopoAlt_values = np.maximum(TopoAlt_interp.values, 0.0)

# # --- Assign back to SparseFXR as a new variable ---
# SparseFXR['feature_centre_topo_alt'] = xr.Variable(
#     dims   = 'sparse_index',
#     data   = TopoAlt_values,
#     attrs  = {
#         'long_name' : 'Topographic altitude beneath feature centre',
#         'units'     : 'm',
#         'source'    : 'GEBCO DEM, bilinear interpolation',
#         'note'      : 'Ocean/below-sea-level values clipped to 0.0 m',
#     }
# )


In [ ]:
SparseFXR

In [ ]:
# CHAD FUNCTION
# creates a base map of topography
def _BuildBasemap(ax, lon_min, lon_max, lat_min, lat_max):
    """
    Add terrain shading, coastlines, and gridlines to an existing
    cartopy axes object.  Returns the axes (modified in place).
    """
    from matplotlib.colors     import ListedColormap, BoundaryNorm
    import matplotlib.ticker   as mticker
    from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
    import cartopy.crs         as ccrs
    import cartopy.feature     as cfeature

    try:
        gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
        dem_da     = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

        if dem_da is None:
            raise ValueError('GEBCO DEM returned None')

        dem_lon  = dem_da.lon.values
        dem_lat  = dem_da.lat.values
        dem_data = dem_da.values

        if dem_lon.ndim == 1 and dem_lat.ndim == 1:
            dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
        else:
            dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

        if not np.any(np.isfinite(dem_data)):
            raise ValueError('DEM has no finite values in this domain')

        colours_topo = [
            '#dde4e8', '#c4dec2', '#e4edc9', '#f3f0cf',
            '#e9d7bd', '#ddc4aa', '#cfb194', '#b58f6e',
        ]
        bounds_topo = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]
        cmap_elev   = ListedColormap(colours_topo)
        norm_topo   = BoundaryNorm(bounds_topo, len(colours_topo), clip=True)

        ax.pcolormesh(
            dem_lon_2d, dem_lat_2d, dem_data,
            cmap=cmap_elev, norm=norm_topo,
            alpha=1.0, transform=ccrs.PlateCarree(),
        )
        ax.contour(
            dem_lon_2d, dem_lat_2d, dem_data,
            levels=[0.0], colors='black', linewidths=0.5,
            transform=ccrs.PlateCarree(), zorder=15,
        )
        ax.contour(
            dem_lon_2d, dem_lat_2d, dem_data,
            levels=[400.0], colors='black', linewidths=0.3,
            transform=ccrs.PlateCarree(), zorder=15,
        )

    except Exception as e:
        print(f'  Terrain shading failed: {e}')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
        ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.3, zorder=2)

    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)

    gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
    gl_minor.xlocator = mticker.MultipleLocator(0.1)
    gl_minor.ylocator = mticker.MultipleLocator(0.1)

    gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
    gl_mid.xlocator = mticker.MultipleLocator(0.5)
    gl_mid.ylocator = mticker.MultipleLocator(0.5)

    gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
    gl_major.xlocator = mticker.MultipleLocator(1.0)
    gl_major.ylocator = mticker.MultipleLocator(1.0)

    for gl in (gl_mid, gl_major):
        gl.xformatter   = LONGITUDE_FORMATTER
        gl.yformatter   = LATITUDE_FORMATTER
        gl.top_labels   = False
        gl.right_labels = False

    return ax

In [ ]:
# CHAD PLOT
# Kernel Density of Whatever Lats/ Lons you give it:

MeanLats = SparseFXR['meanlat'].values
MeanLons = SparseFXR['meanlon'].values

# ── Inputs ─────────────────────────────────────────────────────────────────────
# Replace these with your actual arrays
plot_lats = MeanLats[SparseTrackDeathLandMask * SparseTrackBirthOceanMask]
plot_lons = MeanLons[SparseTrackDeathLandMask * SparseTrackBirthOceanMask]   # 1-D numpy array of longitudes

# ── Filter to valid points ─────────────────────────────────────────────────────
valid     = np.isfinite(plot_lats) & np.isfinite(plot_lons)
plot_lats = plot_lats[valid]
plot_lons = plot_lons[valid]

# ── Map extent ─────────────────────────────────────────────────────────────────
lon_min = float(np.nanmin(plot_lons))
lon_max = float(np.nanmax(plot_lons))
lat_min = float(np.nanmin(plot_lats))
lat_max = float(np.nanmax(plot_lats))

# ── KDE ────────────────────────────────────────────────────────────────────────
lon_grid           = np.linspace(lon_min, lon_max, 300)
lat_grid           = np.linspace(lat_min, lat_max, 300)
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
kde                = gaussian_kde(np.vstack([plot_lons, plot_lats]), bw_method=0.1)
kde_values         = kde(np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])).reshape(lon_mesh.shape)

# Convert to density per 100 km²
mean_lat       = np.mean(plot_lats)
km2_per_deg2   = 111.32 * 111.32 * np.cos(np.radians(mean_lat))
kde_per_100km2 = (kde_values / km2_per_deg2) * len(plot_lats) * 100.0

# ── Colour scale ───────────────────────────────────────────────────────────────
kde_max        = float(np.nanmax(kde_per_100km2))
level_step     = kde_max / 25.0
levels_fill    = np.arange(0, kde_max + level_step, level_step)
levels_visible = levels_fill[1:]

# ── Figure ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(
    figsize    = (8, 6),
    subplot_kw = {'projection': ccrs.PlateCarree()},
    facecolor  = 'white',
)

_BuildBasemap(ax, lon_min, lon_max, lat_min, lat_max)

# invisible contourf to anchor the shared colour scale
ax.contourf(
    lon_mesh, lat_mesh, kde_per_100km2,
    levels    = levels_fill,
    cmap      = 'jet',
    alpha     = 0.0,
    transform = ccrs.PlateCarree(),
    zorder    = 24,
    extend    = 'max',
)
# visible contourf
ax.contourf(
    lon_mesh, lat_mesh, kde_per_100km2,
    levels    = levels_visible,
    cmap      = 'jet',
    alpha     = 0.25,
    transform = ccrs.PlateCarree(),
    zorder    = 24,
    extend    = 'max',
)

# ── Colourbar ──────────────────────────────────────────────────────────────────
plt.draw()  # needed so ax.get_position() is accurate
cbar_ax = fig.add_axes([
    ax.get_position().x1 + 0.025,
    ax.get_position().y0,
    0.02,
    ax.get_position().height,
])
sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=0, vmax=kde_max))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical', extend='max')
cbar.set_label('Feature Locations per 100 km²', fontsize=9)
cbar.set_ticks(np.linspace(0, kde_max, 11))
cbar.set_ticklabels([f'{v:.1f}' for v in np.linspace(0, kde_max, 11)])
cbar.ax.tick_params(labelsize=8)

plt.show()


In [ ]:

FeatureDepths20DBZ = array = np.nan_to_num(SparseFXR['maxETH_20dbz'].values, nan=0) # replace non-existing echo tops with 0 km

MeanDepths20DBZocean = np.mean(FeatureDepths20DBZ[SparseTrackBirthOceanMask * SparseTrackDeathOceanMask * DurationMask])
MeanDepths20DBZland = np.mean(FeatureDepths20DBZ[SparseTrackBirthLandMask * SparseTrackDeathLandMask * DurationMask])
print(MeanDepths20DBZocean)
print(MeanDepths20DBZland)

In [ ]:
SparseFXR

In [ ]:
# CAHD FUNCTION
# prints out ocean-ocean and land-land feature means
def StatByLandOcean(SparseFXR, VarName, Statistic, BirthLandMask, BirthOceanMask, DeathLandMask, DeathOceanMask, DurationMask):
    """
    Computes the mean or median of a variable from SparseFXR for features
    that were born and died entirely over land vs entirely over ocean.

    Parameters
    ----------
    SparseFXR      : xr.Dataset - Sparse tracking dataset
    VarName        : str        - Name of the variable in SparseFXR to average
    Statistic      : str        - 'mean' or 'median'
    BirthLandMask  : np.ndarray - Boolean mask, True where feature was born over land
    BirthOceanMask : np.ndarray - Boolean mask, True where feature was born over ocean
    DeathLandMask  : np.ndarray - Boolean mask, True where feature died over land
    DeathOceanMask : np.ndarray - Boolean mask, True where feature died over ocean
    DurationMask   : np.ndarray - Boolean mask, True where feature meets duration threshold

    Returns
    -------
    StatOcean : float - Statistic of var for fully oceanic features
    StatLand  : float - Statistic of var for fully land features
    """
    if Statistic not in ('mean', 'median'):
        raise ValueError("Statistic must be exactly 'mean' or 'median'")

    # extract variable and replace NaNs with 0 (e.g. non-existing echo tops become 0 km)
    array = np.nan_to_num(SparseFXR[VarName].values, nan=0)

    StatFunc = np.mean if Statistic == 'mean' else np.median

    StatOcean = StatFunc(array[BirthOceanMask * DeathOceanMask * DurationMask])
    StatLand  = StatFunc(array[BirthLandMask  * DeathLandMask  * DurationMask])

    print(f'{VarName}  ({Statistic})')
    print(f'  Ocean : {StatOcean:.4f}')
    print(f'  Land  : {StatLand:.4f}')

    return StatOcean, StatLand


In [ ]:
StatOcean, StatLand = StatByLandOcean(
    SparseFXR      = SparseFXR,
    VarName        = 'max_dbz',
    Statistic      = 'mean',
    BirthLandMask  = SparseTrackBirthLandMask,
    BirthOceanMask = SparseTrackBirthOceanMask,
    DeathLandMask  = SparseTrackDeathLandMask,
    DeathOceanMask = SparseTrackDeathOceanMask,
    DurationMask   = DurationMask,
)


In [ ]:
MeanAreas       = np.mean(FeatureAreas[ComboMask])
MeanDepths20DBZ = np.mean(FeatureDepths20DBZ[ComboMask])
# special case for Z to DBZ
MeanIntensities = np.log10(np.mean(ZIntensities[ComboMask])) * 10
MeanDurations   = np.mean(TrackDurationsInMinutes[ComboMask])             # CAUTION, THIS VARIABLE IS WEIGHTED BY TRACK DURATION
MeanUspeeds     = np.nanmean(Uspeeds[ComboMask])   # WHY ARE THERE NANS HERE????
MeanVspeeds     = np.nanmean(Vspeeds[ComboMask])   # WHY ARE THERE NANS HERE????
# MeanDirections[i]  = np.mean(FeatureDirections[ComboMask])
# MeanSpeeds[i]      = np.mean(FeatureSpeeds[ComboMask])
NumFeatureFrames = np.sum(ComboMask)
NumFeatures = len(np.unique(TrackIDs[ComboMask]))

In [ ]:
# retrieve the opening and ending seconds of the period of feature tracking as pandas time stamps

FirstFrameTimeStamp = pd.to_datetime(SparseFXR.attrs['startdate'], format='%Y%m%d.%H%M%S')
LastFrameTimeStamp = pd.to_datetime(SparseFXR.attrs['enddate'], format='%Y%m%d.%H%M%S')

PeriodStartTimeStamp = FirstFrameTimeStamp
PeriodEndTimeStamp = LastFrameTimeStamp + pd.Timedelta(minutes=5)

print(PeriodStartTimeStamp)
print(PeriodEndTimeStamp)

In [ ]:
HourChunkLengths = 6 # [hours] how long of periods would you like feature summary statistics for?
MinDuration     = 27 # [minutes] minimum number of minutes a feature has to survive for you to consider it

HourChunkEdges = pd.date_range(start=PeriodStartTimeStamp, 
                           end=PeriodEndTimeStamp, 
                           freq= str(HourChunkLengths) + 'h')

NumHourChunks = np.size(HourChunkEdges)-1 # how many X-hour chunks are there in the feature tracking period

# empty arrays the length of the number of hour chunks in the period to fill with means
MeanAreas        = np.full(NumHourChunks, np.nan)
MeanDepths20DBZ  = np.full(NumHourChunks, np.nan)
MeanIntensities  = np.full(NumHourChunks, np.nan)
MeanUspeeds      = np.full(NumHourChunks, np.nan)
MeanVspeeds      = np.full(NumHourChunks, np.nan)
# MeanDirections   = np.full(NumHourChunks, np.nan)
# MeanSpeeds       = np.full(NumHourChunks, np.nan)
MeanDurations    = np.full(NumHourChunks, np.nan) # mean time in minutes a feature lasts
NumFeatureFrames = np.full(NumHourChunks, 0) # how many feature-frames exist in each hour block
NumFeatures      = np.full(NumHourChunks, 0) # how many individual features frames exist in each hour block
NumFeatures      = np.full(NumHourChunks, 0) # how many individual features exist in each hour block

# retrieve feature stat values before entering the loop
BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)

TrackDurationsInFrames  = SparseFXR['track_duration_sparse_indexed'].values
TrackDurationsInMinutes = (TrackDurationsInFrames - 1) * 5
Uspeeds = SparseFXR['Uspeeds'].values
Vspeeds = SparseFXR['Vspeeds'].values

TrackIDs = SparseFXR['tracks_indices'].values


FeatureAreas       = SparseFXR['core_area'].values
FeatureDepths20DBZ = array = np.nan_to_num(SparseFXR['maxETH_20dbz'].values, nan=0) # replace non-existing echo tops with 0 km
# special case for DBZ to Z 
ZIntensities = 10 ** (SparseFXR['max_dbz'].values * 0.1)
# FeatureDirections  = SparseFXR['direction'].values
# FeatureSpeeds      = nSparseFXR['speed'].values


for i in range(0, np.size(HourChunkEdges)-1):
    HourBlockStart = HourChunkEdges[i]
    HourBlockEnd = HourChunkEdges[i+1]

    # select only those track-frames that lie within the time block and belong to features lasting the propper number of frames
    HourBlockMask = (BaseTimesFiveMin >= HourBlockStart) & (BaseTimesFiveMin < HourBlockEnd)
    DurationMask  = (TrackDurationsInMinutes   >= MinDuration)
    ComboMask = HourBlockMask * DurationMask

    TrackIDsInChunkSet =  set(TrackIDs[ComboMask]) # list of each feature the crosses into this time period
    TrackIDsInChunk     = [int(x) for x in TrackIDsInChunkSet] # have to turn the set into integers again

    MeanAreas[i]       = np.mean(FeatureAreas[ComboMask])
    MeanDepths20DBZ[i] = np.mean(FeatureDepths20DBZ[ComboMask])
    # special case for Z to DBZ
    MeanIntensities[i] = np.log10(np.mean(ZIntensities[ComboMask])) * 10
    MeanDurations[i]   = np.mean(TrackDurationsInMinutes[ComboMask])             # CAUTION, THIS VARIABLE IS WEIGHTED BY TRACK DURATION
    MeanUspeeds[i]     = np.nanmean(Uspeeds[ComboMask])   # WHY ARE THERE NANS HERE????
    MeanVspeeds[i]     = np.nanmean(Vspeeds[ComboMask])   # WHY ARE THERE NANS HERE????
    # MeanDirections[i]  = np.mean(FeatureDirections[ComboMask])
    # MeanSpeeds[i]      = np.mean(FeatureSpeeds[ComboMask])
    NumFeatureFrames[i] = np.sum(ComboMask)
    NumFeatures[i] = len(np.unique(TrackIDs[ComboMask]))

    # NumFeatures = np.sum()

    # for TrackID in TrackIDsInChunk:
        
    

    # print(HourBlockStart)

In [ ]:
print(np.sum(np.isnan(MeanAreas)))
print(np.sum(np.isnan(MeanDepths20DBZ)))
print(np.sum(np.isnan(MeanIntensities)))
print(np.sum(np.isnan(MeanDurations)))
print(np.sum(np.isnan(MeanUspeeds)))
print(np.sum(np.isnan(MeanVspeeds)))
print(np.sum(np.isnan(NumFeatures)))



In [ ]:
MeanDepths20DBZ

In [ ]:
# Stack as columns — each array becomes a feature column
data = np.column_stack([MeanAreas, MeanDepths20DBZ, MeanIntensities, MeanDurations, MeanUspeeds, MeanVspeeds, NumFeatures])

In [ ]:
# make each feature have mean=0 and std=1
Scaler = StandardScaler()
DataScaled = Scaler.fit_transform(data)


In [ ]:
KM = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)
KM.fit(DataScaled)

# get the cluster label for each of your 5570 features
ClusterLabels = KM.labels_
print(ClusterLabels)        # e.g. [0, 2, 1, 4, 0, 3, ...]
print(ClusterLabels.shape)  # (5570,)

In [ ]:
# TRY TO APPLY THE ELBOW METHOD USING INERTIA!!!
# CHAD PLOT

inertias = []
k_range  = range(1, 15)

for k in k_range:
    KM = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    KM.fit(DataScaled)
    inertias.append(KM.inertia_)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(k_range, inertias, marker='o', color='steelblue', linewidth=2, markersize=6)

ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (Within-Cluster Sum of Squares)')
ax.set_title('Elbow Method — Optimal Number of Clusters')
ax.set_xticks(k_range)

plt.tight_layout()
plt.show()

In [ ]:
# TRY TO APPLY THE ELBOW METHOD USING A "SILHOUETTE SCORE"!!!
# CHAD PLOT

from sklearn.metrics import silhouette_score

silhouette_scores = []
k_range           = range(2, 15)   # silhouette is undefined for k=1

for k in k_range:
    KM = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    KM.fit(DataScaled)
    silhouette_scores.append(silhouette_score(DataScaled, KM.labels_))

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(k_range, silhouette_scores, marker='o', color='steelblue', linewidth=2, markersize=6)

ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score — Optimal Number of Clusters')
ax.set_xticks(k_range)

plt.tight_layout()
plt.show()


In [ ]:
# CHAD PLOT
# ALL 3 VARIABLES
# ECHO TOP HEIGHT, DBZ, AREA

Fig = plt.figure(figsize=(10, 8))
Ax  = Fig.add_subplot(111, projection='3d')

# get unique clusters and assign a colour to each
UniqueClusters = np.unique(ClusterLabels)
Cmap           = plt.cm.get_cmap('tab10', len(UniqueClusters))

for i, Cluster in enumerate(UniqueClusters):
    Mask = (ClusterLabels == Cluster)
    Ax.scatter(
        MeanAreas[Mask],
        MeanUspeeds[Mask],
        NumFeatures[Mask],
        color=Cmap(i),
        s=10,
        alpha=0.6,
        label=f'Cluster {Cluster}'
    )

Ax.set_xlim(0,  140)  # area in km^2
# Ax.set_ylim(0, 55)   # reflectivity in dBZ
# Ax.set_zlim(0,  12)   # ETH in kilometres

Ax.set_xlabel('Mean Area [km^2]')
Ax.set_ylabel('Mean U Speed [m/s]')
Ax.set_zlabel('Number of Features')
Ax.set_title('K-Means Clustering of Mackay Feature Tracks 2024-03-09')

Ax.legend(title='Cluster', bbox_to_anchor=(1.15, 1), loc='upper left')
plt.tight_layout()

# SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/'
# SaveFile   = 'KmeansClusteringFeatures_22_20240214_3D.png'

# SavePath = SaveFolder + SaveFile

# if not Path(SaveFolder).exists():
#     print('doing')
#     Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)

In [ ]:
# ATTEMPTING SELF-ORGANISING MAPS:

# --- Define grid size ---
# A common rule of thumb for grid size is:
#   side length ≈ sqrt(sqrt(n_samples))
# For 5570 samples: sqrt(sqrt(5570)) ≈ 8.6, so an 8x8 or 9x9 grid is reasonable
grid_x = 2
grid_y = 2

# --- Initialise SOM ---
# input_len must match the number of features (columns) in DataScaled
SOM = MiniSom(
    x              = grid_x,
    y              = grid_y,
    input_len      = DataScaled.shape[1],
    sigma          = 1.0,       # neighbourhood radius — how far influence spreads
    learning_rate  = 0.5,       # how much nodes update each step
    random_seed    = 42,
)

# --- Initialise weights using PCA (more stable than random initialisation) ---
SOM.pca_weights_init(DataScaled)

# --- Train ---
SOM.train(
    DataScaled,
    num_iteration  = 10000,
    verbose        = True,
)

# --- Get the best matching unit (BMU) for each sample ---
# This is analogous to KM.labels_ — it tells you which grid node
# each of your 5570 features was assigned to
BMUs = np.array([SOM.winner(x) for x in DataScaled])
print(BMUs)          # e.g. [(3,2), (7,1), (0,4), ...]
print(BMUs.shape)    # (5570, 2)

# Convert (x, y) grid positions to flat cluster labels if needed
# (analogous to KM.labels_)
ClusterLabels_SOM = np.array([bmu[0] * grid_y + bmu[1] for bmu in BMUs])
print(ClusterLabels_SOM)        # e.g. [26, 57, 4, ...]
print(ClusterLabels_SOM.shape)  # (5570,)


In [ ]:
# ATTEMPTING SELF-ORGANISING MAPS: VIEWING THE RESULTS:

fig, ax = plt.subplots(figsize=(8, 8))

# U-matrix: mean distance from each node to its neighbours
umatrix = SOM.distance_map()

im = ax.imshow(umatrix.T, cmap='bone_r', origin='lower')
plt.colorbar(im, ax=ax, label='Mean Distance to Neighbours')

ax.set_title('SOM U-Matrix')
ax.set_xlabel('Grid X')
ax.set_ylabel('Grid Y')

plt.tight_layout()
plt.show()


In [ ]:
quantisation_errors = []
grid_sizes          = range(2, 12)

for g in grid_sizes:
    SOM_test = MiniSom(
        x             = g,
        y             = g,
        input_len     = DataScaled.shape[1],
        sigma         = 1.0,
        learning_rate = 0.5,
        random_seed   = 42,
    )
    SOM_test.pca_weights_init(DataScaled)
    SOM_test.train(DataScaled, num_iteration=5000, verbose=False)
    quantisation_errors.append(SOM_test.quantization_error(DataScaled))

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    [g*g for g in grid_sizes],
    quantisation_errors,
    marker='o', color='steelblue', linewidth=2, markersize=6,
)

ax.set_xlabel('Number of Nodes (Grid Size²)')
ax.set_ylabel('Quantisation Error')
ax.set_title('SOM — Choosing Grid Size')

plt.tight_layout()
plt.show()


In [ ]:
counts, bin_edges = np.histogram(MeanVspeeds, bins=30)

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    bin_edges[:-1],
    counts,
    width  = np.diff(bin_edges),
    align  = 'edge',
    color  = 'steelblue',
    edgecolor = 'white',
    linewidth = 0.5,
)

ax.set_xlabel('Number of Feature Frames')
ax.set_ylabel('Count')
ax.set_title('Distribution of Features')

plt.tight_layout()
plt.show()


In [ ]:
TrackIDsInChunk

In [ ]:
 NumFeatureFrames

In [ ]:
BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)

np.min(BaseTimesFiveMin)

In [ ]:
for i in range(0,100):

    print('sparse index: ' + str(SparseFXR['sparse_index'].values[i]))
    
    print('track: ' + str(SparseFXR['tracks_indices'].values[i]))
    
    print('frame: ' + str(SparseFXR['times_indices'].values[i]))
    
    print('time: ' + str(SparseFXR['base_time'].values[i]))

    print('')



In [ ]:
# CHAD PLOT
# SPARSE LOADED VERSION
# FEATURE LOCATIONS (BIRTH DEATH AND ALL) AND KERENEL DENSITY

def PlotFeatureLocationsSparse(
    SparseFXR,
    RadarIDno,
    RadarSiteName,
    QualityControlOption,
    DateRange,
    TimeRange,
    DotsIncluded,
    MinDuration,
):
    """
    Plots feature locations (birth, death, or all frames) from a pre-loaded
    sparse PyFLEXTRKR xarray Dataset (SparseFXR).

    Produces two plots:
        1. Scatter plot  — small red dots on terrain background
        2. Density plot  — kernel density estimate on terrain background

    Parameters
    ----------
    SparseFXR            : xr.Dataset - Pre-loaded sparse tracking dataset
    RadarIDno            : str        - Radar ID number string (e.g. '22')
    RadarSiteName        : str        - Radar site name for plot title
    QualityControlOption : int        - QC option number
    DateRange            : str        - 'YYYYMMDD-YYYYMMDD' inclusive date range
    TimeRange            : str        - 'HH:MM-HH:MM' UTC time-of-day window
    DotsIncluded         : str        - 'Birth', 'Death', or 'All'
    MinDuration          : float      - Minimum feature duration in minutes
    RangeStartHour       : int        - Start hour of time window (UTC)
    RangeEndHour         : int        - End hour of time window (UTC)
    """

    # ── Parse time range ───────────────────────────────────────────────────────

    time_start_str, time_end_str = TimeRange.split('-')
    RangeStartHour  = int(time_start_str.split(':')[0])
    RangeStartMin   = int(time_start_str.split(':')[1])
    RangeEndHour    = int(time_end_str.split(':')[0])
    RangeEndMin     = int(time_end_str.split(':')[1])
    
    RangeStartMins  = RangeStartHour * 60 + RangeStartMin
    RangeEndMins    = RangeEndHour   * 60 + RangeEndMin

    # ── Input validation ───────────────────────────────────────────────────────

    if DotsIncluded not in ('Birth', 'Death', 'All'):
        raise ValueError("DotsIncluded must be 'Birth', 'Death', or 'All'.")

    # ── Parse date range for titles / save paths ───────────────────────────────

    date_start_str = DateRange[:8]
    date_end_str   = DateRange[9:]
    date_start     = pd.Timestamp(f'{date_start_str[:4]}-{date_start_str[4:6]}-{date_start_str[6:8]}')
    date_end       = pd.Timestamp(f'{date_end_str[:4]}-{date_end_str[4:6]}-{date_end_str[6:8]}')

    # ── Build masks ────────────────────────────────────────────────────────────

    MinFrames      = int(np.ceil(MinDuration / 5) + 1)

    BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)
    FeatureFrames    = SparseFXR['times_indices'].values
    TrackDurations   = SparseFXR['track_duration_sparse_indexed'].values

    TimeMask = (
        (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute >= RangeStartMins) &
        (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute <  RangeEndMins)
    )


    BirthMask    = (FeatureFrames == 0)
    DeathMask    = (FeatureFrames == TrackDurations - 1)
    DurationMask = (TrackDurations >= MinFrames)

    # ── Select lat/lon based on DotsIncluded ──────────────────────────────────

    if DotsIncluded == 'Birth':
        ActiveMask = TimeMask & DurationMask & BirthMask
    elif DotsIncluded == 'Death':
        ActiveMask = TimeMask & DurationMask & DeathMask
    else:  # 'All'
        ActiveMask = TimeMask & DurationMask

    all_lats = SparseFXR['core_meanlat'].values[ActiveMask]
    all_lons = SparseFXR['core_meanlon'].values[ActiveMask]

    # ── Drop any NaN lat/lon ───────────────────────────────────────────────────

    valid = np.isfinite(all_lats) & np.isfinite(all_lons)
    all_lats = all_lats[valid]
    all_lons = all_lons[valid]

    print(f"Duration filter : >= {MinDuration} min  →  >= {MinFrames} frames")
    print(f"Date range      : {date_start.date()} to {date_end.date()}")
    print(f"Time window     : {TimeRange} UTC")
    print(f"Mode            : {DotsIncluded}")
    print(f"Points collected: {len(all_lats)}")

    if len(all_lats) == 0:
        print("No valid points found — nothing to plot.")
        return

    # ── Shared map extent ──────────────────────────────────────────────────────

    lon_min = float(np.nanmin(all_lons))
    lon_max = float(np.nanmax(all_lons))
    lat_min = float(np.nanmin(all_lats))
    lat_max = float(np.nanmax(all_lats))

    # ── Shared title suffix ────────────────────────────────────────────────────

    title_suffix = (
        f'{RadarSiteName} Radar  |  {DotsIncluded} Locations\n'
        f'{date_start.strftime("%Y-%m-%d")} to {date_end.strftime("%Y-%m-%d")}  |  '
        f'{TimeRange} UTC  |  Min Duration: {MinDuration} min  |  '
        f'({len(all_lats)} points)'
    )

    # ── Save folder ────────────────────────────────────────────────────────────

    SaveFolder = (
        f'/scratch/v46/sg3241/tmp/pngImages/FeatureLocations/'
        f'{RadarIDno}/{DateRange}/'
    )
    if not Path(SaveFolder).exists():
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)

    SaveBase = (
        f'{RadarIDno}_{DateRange}_{TimeRange.replace(":", "").replace("-", "_")}_'
        f'{DotsIncluded}_{int(MinDuration)}minMin'
    )

        # ── Load first radar file to get radar location ────────────────────────────
    
    RadarSiteName, LonShift = GrabRadarInfo(RadarIDno)
    
    first_date_str  = DateRange[:8]
    radar_file_path = (
        f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
        f'{first_date_str}/QC2/'
        f'{RadarIDno}_{first_date_str}_000000_QC2.nc'
    )
    
    # Fall back to first available file if 000000 is missing
    if not os.path.exists(radar_file_path):
        fallback = sorted(glob.glob(
            f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
            f'{first_date_str}/QC2/{RadarIDno}_{first_date_str}_*_QC2.nc'
        ))
        if len(fallback) == 0:
            raise FileNotFoundError(
                f'No radar grid files found for {first_date_str} under RadarID {RadarIDno}.'
            )
        radar_file_path = fallback[0]
        print(f'  000000 not found — using fallback: {os.path.basename(radar_file_path)}')
    
    RadarXR_ref    = xr.open_dataset(radar_file_path)
    RadarLon       = float(RadarXR_ref.radar_longitude[0]) + LonShift
    RadarLat       = float(RadarXR_ref.radar_latitude[0])
    RadarXR_ref.close()
    
    print(f'  Radar location : {RadarLat:.4f}°N  {RadarLon:.4f}°E')


    # ── Helper: build base map ─────────────────────────────────────────────────

    def build_basemap():

        fig, ax = plt.subplots(
            figsize    = (8, 6),
            subplot_kw = {'projection': ccrs.PlateCarree()},
            facecolor  = 'white',
        )

        # --- Terrain background ---
        try:
            gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
            dem_da     = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

            if dem_da is None:
                raise ValueError('GEBCO DEM returned None')

            dem_lon  = dem_da.lon.values
            dem_lat  = dem_da.lat.values
            dem_data = dem_da.values

            if dem_lon.ndim == 1 and dem_lat.ndim == 1:
                dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
            else:
                dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

            if not np.any(np.isfinite(dem_data)):
                raise ValueError('DEM has no finite values in this domain')

            colours_topo = [
                '#dde4e8', '#c4dec2', '#e4edc9', '#f3f0cf',
                '#e9d7bd', '#ddc4aa', '#cfb194', '#b58f6e',
            ]
            bounds_topo = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]
            cmap_elev   = ListedColormap(colours_topo)
            norm_topo   = BoundaryNorm(bounds_topo, len(colours_topo), clip=True)

            ax.pcolormesh(
                dem_lon_2d, dem_lat_2d, dem_data,
                cmap      = cmap_elev,
                norm      = norm_topo,
                alpha     = 1.0,
                transform = ccrs.PlateCarree(),
            )
            ax.contour(
                dem_lon_2d, dem_lat_2d, dem_data,
                levels    = [0.0],
                colors    = 'black',
                linewidths= 0.5,
                transform = ccrs.PlateCarree(),
                zorder    = 15,
            )
            ax.contour(
                dem_lon_2d, dem_lat_2d, dem_data,
                levels    = [400.0],
                colors    = 'black',
                linewidths= 0.3,
                transform = ccrs.PlateCarree(),
                zorder    = 15,
            )
            print('  Terrain shading loaded successfully.')

        except Exception as e:
            print(f'  Terrain shading failed: {e}')
            ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
            ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.3, zorder=2)

        ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)

        # --- Gridlines ---
        gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
        gl_minor.xlocator = mticker.MultipleLocator(0.1)
        gl_minor.ylocator = mticker.MultipleLocator(0.1)

        gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
        gl_mid.xlocator = mticker.MultipleLocator(0.5)
        gl_mid.ylocator = mticker.MultipleLocator(0.5)

        gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
        gl_major.xlocator = mticker.MultipleLocator(1.0)
        gl_major.ylocator = mticker.MultipleLocator(1.0)

        for gl in (gl_mid, gl_major):
            gl.xformatter   = LONGITUDE_FORMATTER
            gl.yformatter   = LATITUDE_FORMATTER
            gl.top_labels   = False
            gl.right_labels = False

        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

            # --- Radar location star ---
        ax.plot(RadarLon, RadarLat,
                marker='*', color='black', markersize=8,
                transform=ccrs.PlateCarree(), zorder=30)
        ax.plot(RadarLon, RadarLat,
                marker='*', color='white', markersize=4,
                transform=ccrs.PlateCarree(), zorder=31)

        return fig, ax

    # ══════════════════════════════════════════════════════════════════════════
    # PLOT 1 — Scatter (red dots)
    # ══════════════════════════════════════════════════════════════════════════

    print('\nGenerating scatter plot...')
    fig1, ax1 = build_basemap()

    ax1.scatter(
        all_lons, all_lats,
        s          = 6,
        color      = 'red',
        alpha      = 0.6,
        transform  = ccrs.PlateCarree(),
        zorder     = 25,
        edgecolors = 'none',
    )

    ax1.set_title(f'Feature {title_suffix}', fontsize=10)

    plt.tight_layout()
    SavePath1 = SaveFolder + SaveBase + '_Scatter.png'
    # plt.savefig(SavePath1, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    # print(f'  Scatter plot saved to: {SavePath1}')

    # ══════════════════════════════════════════════════════════════════════════
    # PLOT 2 — Kernel Density
    # ══════════════════════════════════════════════════════════════════════════

    print('\nGenerating density plot...')
    fig2, ax2 = build_basemap()

    xy          = np.vstack([all_lons, all_lats])
    kde         = gaussian_kde(xy, bw_method=0.1)

    lon_grid    = np.linspace(lon_min, lon_max, 300)
    lat_grid    = np.linspace(lat_min, lat_max, 300)
    lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
    grid_coords = np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])

    kde_values  = kde(grid_coords).reshape(lon_mesh.shape)

    # --- Convert to points per 100 km² ---
    mean_lat       = np.mean(all_lats)
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.radians(mean_lat))
    km2_per_deg2   = km_per_deg_lat * km_per_deg_lon
    n_points       = len(all_lats)
    kde_per_100km2 = (kde_values / km2_per_deg2) * n_points * 100.0

    # --- Dynamic colour scale ---
    kde_max        = float(np.nanmax(kde_per_100km2))
    level_step     = kde_max / 25.0
    levels_fill    = np.arange(0, kde_max + level_step, level_step)
    levels_visible = levels_fill[1:]

    kde_fill = ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels    = levels_fill,
        cmap      = 'jet',
        alpha     = 0.0,
        transform = ccrs.PlateCarree(),
        zorder    = 24,
        extend    = 'max',
    )

    ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels    = levels_visible,
        cmap      = 'jet',
        alpha     = 0.25,
        transform = ccrs.PlateCarree(),
        zorder    = 24,
        extend    = 'max',
    )

    # --- Colourbar ---
    cax = fig2.add_axes([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])

    sm = plt.cm.ScalarMappable(
        cmap = 'jet',
        norm = plt.Normalize(vmin=0, vmax=kde_max),
    )
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=cax, orientation='vertical', extend='max')
    cbar.set_label('Feature Locations per 100 km²', fontsize=9)

    tick_vals = np.linspace(0, kde_max, 11)
    cbar.set_ticks(tick_vals)
    cbar.set_ticklabels([f'{v:.1f}' for v in tick_vals])
    cbar.ax.tick_params(labelsize=8)

    ax2.set_title(f'Feature Density {title_suffix}', fontsize=10)

    plt.tight_layout()
    plt.draw()

    cax.set_position([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])

    SavePath2 = SaveFolder + SaveBase + '_Density.png'
    # plt.savefig(SavePath2, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    # print(f'  Density plot saved to: {SavePath2}')


In [ ]:
PlotFeatureLocationsSparse(
    SparseFXR            = SparseFXR,
    RadarIDno            = '41',
    RadarSiteName        = 'Willis Island',
    QualityControlOption = 2,
    DateRange            = '20240201-20240229',
    TimeRange            = '04:00-06:00',
    DotsIncluded         = 'All',
    MinDuration          = 30,
)



In [ ]:
# MANY CHAD FUNCTIONS
# USED TO CREATE GRID OF KERNEL DENSITY PLOTS

def _BuildFeatureMasks(SparseFXR, TimeRange, MinDuration):
    """
    Shared pre-work: parse TimeRange, build all masks, return components
    needed by both the single-plot and grid-plot functions.

    Returns
    -------
    dict with keys:
        BirthComboMask, DeathComboMask, AllComboMask,
        RangeStartMins, RangeEndMins, MinFrames
    """
    # ── Parse time range ───────────────────────────────────────────────────
    time_start_str, time_end_str = TimeRange.split('-')
    RangeStartHour = int(time_start_str.split(':')[0])
    RangeStartMin  = int(time_start_str.split(':')[1])
    RangeEndHour   = int(time_end_str.split(':')[0])
    RangeEndMin    = int(time_end_str.split(':')[1])
    RangeStartMins = RangeStartHour * 60 + RangeStartMin
    RangeEndMins   = RangeEndHour   * 60 + RangeEndMin

    # ── Duration filter ────────────────────────────────────────────────────
    MinFrames      = int(np.ceil(MinDuration / 5) + 1)

    # ── Build masks ────────────────────────────────────────────────────────
    BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)
    FeatureFrames    = SparseFXR['times_indices'].values
    TrackDurations   = SparseFXR['track_duration_sparse_indexed'].values

    TimeMask = (
        (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute >= RangeStartMins) &
        (BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute <  RangeEndMins)
    )
    BirthMask    = (FeatureFrames == 0)
    DeathMask    = (FeatureFrames == TrackDurations - 1)
    DurationMask = (TrackDurations >= MinFrames)

    return {
        'BirthComboMask' : TimeMask & DurationMask & BirthMask,
        'DeathComboMask' : TimeMask & DurationMask & DeathMask,
        'AllComboMask'   : TimeMask & DurationMask,
        'RangeStartMins' : RangeStartMins,
        'RangeEndMins'   : RangeEndMins,
        'MinFrames'      : MinFrames,
    }


def _BuildBasemap(ax, lon_min, lon_max, lat_min, lat_max):
    """
    Add terrain shading, coastlines, and gridlines to an existing
    cartopy axes object.  Returns the axes (modified in place).
    """
    from matplotlib.colors     import ListedColormap, BoundaryNorm
    import matplotlib.ticker   as mticker
    from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
    import cartopy.crs         as ccrs
    import cartopy.feature     as cfeature

    try:
        gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
        dem_da     = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

        if dem_da is None:
            raise ValueError('GEBCO DEM returned None')

        dem_lon  = dem_da.lon.values
        dem_lat  = dem_da.lat.values
        dem_data = dem_da.values

        if dem_lon.ndim == 1 and dem_lat.ndim == 1:
            dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
        else:
            dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

        if not np.any(np.isfinite(dem_data)):
            raise ValueError('DEM has no finite values in this domain')

        colours_topo = [
            '#dde4e8', '#c4dec2', '#e4edc9', '#f3f0cf',
            '#e9d7bd', '#ddc4aa', '#cfb194', '#b58f6e',
        ]
        bounds_topo = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]
        cmap_elev   = ListedColormap(colours_topo)
        norm_topo   = BoundaryNorm(bounds_topo, len(colours_topo), clip=True)

        ax.pcolormesh(
            dem_lon_2d, dem_lat_2d, dem_data,
            cmap=cmap_elev, norm=norm_topo,
            alpha=1.0, transform=ccrs.PlateCarree(),
        )
        ax.contour(
            dem_lon_2d, dem_lat_2d, dem_data,
            levels=[0.0], colors='black', linewidths=0.5,
            transform=ccrs.PlateCarree(), zorder=15,
        )
        ax.contour(
            dem_lon_2d, dem_lat_2d, dem_data,
            levels=[400.0], colors='black', linewidths=0.3,
            transform=ccrs.PlateCarree(), zorder=15,
        )

    except Exception as e:
        print(f'  Terrain shading failed: {e}')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
        ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.3, zorder=2)

    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)

    gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
    gl_minor.xlocator = mticker.MultipleLocator(0.1)
    gl_minor.ylocator = mticker.MultipleLocator(0.1)

    gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
    gl_mid.xlocator = mticker.MultipleLocator(0.5)
    gl_mid.ylocator = mticker.MultipleLocator(0.5)

    gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
    gl_major.xlocator = mticker.MultipleLocator(1.0)
    gl_major.ylocator = mticker.MultipleLocator(1.0)

    for gl in (gl_mid, gl_major):
        gl.xformatter   = LONGITUDE_FORMATTER
        gl.yformatter   = LATITUDE_FORMATTER
        gl.top_labels   = False
        gl.right_labels = False

    return ax


def PlotFeatureLocationsSparse(
    SparseFXR,
    RadarIDno,
    RadarSiteName,
    QualityControlOption,
    DateRange,
    TimeRange,
    DotsIncluded,
    MinDuration,
):
    """
    Plots feature locations (birth, death, or all frames) from a pre-loaded
    sparse PyFLEXTRKR xarray Dataset (SparseFXR).

    Produces two plots:
        1. Scatter plot  — small red dots on terrain background
        2. Density plot  — kernel density estimate on terrain background

    Parameters
    ----------
    SparseFXR            : xr.Dataset - Pre-loaded sparse tracking dataset
    RadarIDno            : str        - Radar ID number string (e.g. '22')
    RadarSiteName        : str        - Radar site name for plot title
    QualityControlOption : int        - QC option number
    DateRange            : str        - 'YYYYMMDD-YYYYMMDD' inclusive date range
    TimeRange            : str        - 'HH:MM-HH:MM' UTC time-of-day window
    DotsIncluded         : str        - 'Birth', 'Death', or 'All'
    MinDuration          : float      - Minimum feature duration in minutes
    """

    import pandas as pd
    from scipy.stats           import gaussian_kde
    from pathlib               import Path
    import cartopy.crs         as ccrs
    import matplotlib.ticker   as mticker

    # ── Input validation ───────────────────────────────────────────────────
    if DotsIncluded not in ('Birth', 'Death', 'All'):
        raise ValueError("DotsIncluded must be 'Birth', 'Death', or 'All'.")

    # ── Parse date range ───────────────────────────────────────────────────
    date_start_str = DateRange[:8]
    date_end_str   = DateRange[9:]
    date_start     = pd.Timestamp(f'{date_start_str[:4]}-{date_start_str[4:6]}-{date_start_str[6:8]}')
    date_end       = pd.Timestamp(f'{date_end_str[:4]}-{date_end_str[4:6]}-{date_end_str[6:8]}')

    # ── Build masks ────────────────────────────────────────────────────────
    masks     = _BuildFeatureMasks(SparseFXR, TimeRange, MinDuration)
    MinFrames = masks['MinFrames']

    if DotsIncluded == 'Birth':
        ActiveMask = masks['BirthComboMask']
    elif DotsIncluded == 'Death':
        ActiveMask = masks['DeathComboMask']
    else:
        ActiveMask = masks['AllComboMask']

    all_lats = SparseFXR['core_meanlat'].values[ActiveMask]
    all_lons = SparseFXR['core_meanlon'].values[ActiveMask]

    valid    = np.isfinite(all_lats) & np.isfinite(all_lons)
    all_lats = all_lats[valid]
    all_lons = all_lons[valid]

    print(f"Duration filter : >= {MinDuration} min  →  >= {MinFrames} frames")
    print(f"Date range      : {date_start.date()} to {date_end.date()}")
    print(f"Time window     : {TimeRange} UTC")
    print(f"Mode            : {DotsIncluded}")
    print(f"Points collected: {len(all_lats)}")

    if len(all_lats) == 0:
        print("No valid points found — nothing to plot.")
        return

    # ── Map extent ─────────────────────────────────────────────────────────
    lon_min = float(np.nanmin(all_lons))
    lon_max = float(np.nanmax(all_lons))
    lat_min = float(np.nanmin(all_lats))
    lat_max = float(np.nanmax(all_lats))

    # ── Radar location ─────────────────────────────────────────────────────
    _, LonShift     = GrabRadarInfo(RadarIDno)
    first_date_str  = DateRange[:8]
    radar_file_path = (
        f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
        f'{first_date_str}/QC2/{RadarIDno}_{first_date_str}_000000_QC2.nc'
    )
    if not os.path.exists(radar_file_path):
        fallback = sorted(glob.glob(
            f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
            f'{first_date_str}/QC2/{RadarIDno}_{first_date_str}_*_QC2.nc'
        ))
        if len(fallback) == 0:
            raise FileNotFoundError(
                f'No radar grid files found for {first_date_str} under RadarID {RadarIDno}.'
            )
        radar_file_path = fallback[0]
        print(f'  000000 not found — using fallback: {os.path.basename(radar_file_path)}')

    RadarXR_ref = xr.open_dataset(radar_file_path)
    RadarLon    = float(RadarXR_ref.radar_longitude[0]) + LonShift
    RadarLat    = float(RadarXR_ref.radar_latitude[0])
    RadarXR_ref.close()
    print(f'  Radar location : {RadarLat:.4f}°N  {RadarLon:.4f}°E')

    # ── Title / save ───────────────────────────────────────────────────────
    title_suffix = (
        f'{RadarSiteName} Radar  |  {DotsIncluded} Locations\n'
        f'{date_start.strftime("%Y-%m-%d")} to {date_end.strftime("%Y-%m-%d")}  |  '
        f'{TimeRange} UTC  |  Min Duration: {MinDuration} min  |  '
        f'({len(all_lats)} points)'
    )

    SaveFolder = (
        f'/scratch/v46/sg3241/tmp/pngImages/FeatureLocations/'
        f'{RadarIDno}/{DateRange}/'
    )
    if not Path(SaveFolder).exists():
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)

    SaveBase = (
        f'{RadarIDno}_{DateRange}_{TimeRange.replace(":", "").replace("-", "_")}_'
        f'{DotsIncluded}_{int(MinDuration)}minMin'
    )

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 1 — Scatter
    # ══════════════════════════════════════════════════════════════════════

    print('\nGenerating scatter plot...')
    fig1, ax1 = plt.subplots(
        figsize    = (8, 6),
        subplot_kw = {'projection': ccrs.PlateCarree()},
        facecolor  = 'white',
    )
    _BuildBasemap(ax1, lon_min, lon_max, lat_min, lat_max)

    ax1.scatter(
        all_lons, all_lats,
        s=6, color='red', alpha=0.6,
        transform=ccrs.PlateCarree(), zorder=25, edgecolors='none',
    )
    ax1.plot(RadarLon, RadarLat, marker='*', color='black', markersize=8,
             transform=ccrs.PlateCarree(), zorder=30)
    ax1.plot(RadarLon, RadarLat, marker='*', color='white', markersize=4,
             transform=ccrs.PlateCarree(), zorder=31)

    ax1.set_title(f'Feature {title_suffix}', fontsize=10)
    plt.tight_layout()
    SavePath1 = SaveFolder + SaveBase + '_Scatter.png'
    # plt.savefig(SavePath1, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    print(f'  Scatter plot saved to: {SavePath1}')

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 2 — Kernel Density
    # ══════════════════════════════════════════════════════════════════════

    print('\nGenerating density plot...')
    fig2, ax2 = plt.subplots(
        figsize    = (8, 6),
        subplot_kw = {'projection': ccrs.PlateCarree()},
        facecolor  = 'white',
    )
    _BuildBasemap(ax2, lon_min, lon_max, lat_min, lat_max)

    xy             = np.vstack([all_lons, all_lats])
    kde            = gaussian_kde(xy, bw_method=0.1)
    lon_grid       = np.linspace(lon_min, lon_max, 300)
    lat_grid       = np.linspace(lat_min, lat_max, 300)
    lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
    kde_values     = kde(np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])).reshape(lon_mesh.shape)

    mean_lat       = np.mean(all_lats)
    km2_per_deg2   = 111.32 * 111.32 * np.cos(np.radians(mean_lat))
    kde_per_100km2 = (kde_values / km2_per_deg2) * len(all_lats) * 100.0

    kde_max        = float(np.nanmax(kde_per_100km2))
    level_step     = kde_max / 25.0
    levels_fill    = np.arange(0, kde_max + level_step, level_step)
    levels_visible = levels_fill[1:]

    ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels=levels_fill, cmap='jet', alpha=0.0,
        transform=ccrs.PlateCarree(), zorder=24, extend='max',
    )
    ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels=levels_visible, cmap='jet', alpha=0.25,
        transform=ccrs.PlateCarree(), zorder=24, extend='max',
    )

    ax2.plot(RadarLon, RadarLat, marker='*', color='black', markersize=8,
             transform=ccrs.PlateCarree(), zorder=30)
    ax2.plot(RadarLon, RadarLat, marker='*', color='white', markersize=4,
             transform=ccrs.PlateCarree(), zorder=31)

    cax = fig2.add_axes([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])
    sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=0, vmax=kde_max))
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=cax, orientation='vertical', extend='max')
    cbar.set_label('Feature Locations per 100 km²', fontsize=9)
    tick_vals = np.linspace(0, kde_max, 11)
    cbar.set_ticks(tick_vals)
    cbar.set_ticklabels([f'{v:.1f}' for v in tick_vals])
    cbar.ax.tick_params(labelsize=8)

    ax2.set_title(f'Feature Density {title_suffix}', fontsize=10)
    plt.tight_layout()
    plt.draw()
    cax.set_position([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])

    SavePath2 = SaveFolder + SaveBase + '_Density.png'
    # plt.savefig(SavePath2, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    # print(f'  Density plot saved to: {SavePath2}')


def PlotFeatureDensityGrid(
    SparseFXR,
    RadarIDno,
    RadarSiteName,
    QualityControlOption,
    DateRange,
    MinDuration,
):
    """
    Produces three 4×2 grid figures of kernel density plots — one each for
    Birth, Death, and All — with one panel per 3-hour UTC block.

    All 8 panels within a figure share the same colour scale (normalised to
    the global maximum across all bins) and the same map extent (union of
    all points across all bins).

    Parameters
    ----------
    SparseFXR            : xr.Dataset - Pre-loaded sparse tracking dataset
    RadarIDno            : str        - Radar ID number string
    RadarSiteName        : str        - Radar site name for plot title
    QualityControlOption : int        - QC option number
    DateRange            : str        - 'YYYYMMDD-YYYYMMDD'
    MinDuration          : float      - Minimum feature duration in minutes
    """

    import pandas as pd
    from scipy.stats  import gaussian_kde
    from pathlib      import Path
    import cartopy.crs as ccrs

    # ── 3-hour time bins ───────────────────────────────────────────────────
    time_bins = [
        ('00:00', '03:00'),
        ('03:00', '06:00'),
        ('06:00', '09:00'),
        ('09:00', '12:00'),
        ('12:00', '15:00'),
        ('15:00', '18:00'),
        ('18:00', '21:00'),
        ('21:00', '24:00'),
    ]

    # ── Parse date range for titles ────────────────────────────────────────
    date_start_str = DateRange[:8]
    date_end_str   = DateRange[9:]
    date_start     = pd.Timestamp(f'{date_start_str[:4]}-{date_start_str[4:6]}-{date_start_str[6:8]}')
    date_end       = pd.Timestamp(f'{date_end_str[:4]}-{date_end_str[4:6]}-{date_end_str[6:8]}')

    MinFrames = int(np.ceil(MinDuration / 5) + 1)

    # ── Radar location ─────────────────────────────────────────────────────
    _, LonShift     = GrabRadarInfo(RadarIDno)
    radar_file_path = (
        f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
        f'{date_start_str}/QC2/{RadarIDno}_{date_start_str}_000000_QC2.nc'
    )
    if not os.path.exists(radar_file_path):
        fallback = sorted(glob.glob(
            f'/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/{RadarIDno}/'
            f'{date_start_str}/QC2/{RadarIDno}_{date_start_str}_*_QC2.nc'
        ))
        if len(fallback) == 0:
            raise FileNotFoundError(
                f'No radar grid files found for {date_start_str} under RadarID {RadarIDno}.'
            )
        radar_file_path = fallback[0]
        print(f'  000000 not found — using fallback: {os.path.basename(radar_file_path)}')

    RadarXR_ref = xr.open_dataset(radar_file_path)
    RadarLon    = float(RadarXR_ref.radar_longitude[0]) + LonShift
    RadarLat    = float(RadarXR_ref.radar_latitude[0])
    RadarXR_ref.close()
    # print(f'  Radar location : {RadarLat:.4f}°N  {RadarLon:.4f}°E')

    # ── Save folder ────────────────────────────────────────────────────────
    SaveFolder = (
        f'/scratch/v46/sg3241/tmp/pngImages/FeatureLocations/'
        f'{RadarIDno}/{DateRange}/'
    )
    if not Path(SaveFolder).exists():
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)

    SaveBase = (
        f'{RadarIDno}_{DateRange}_000000_240000_3hourblocks_{int(MinDuration)}minMin'
    )

    # ── Pre-compute base arrays used for masking ───────────────────────────
    BaseTimesFiveMin = pd.DatetimeIndex(SparseFXR['base_time_fivemin'].values)
    FeatureFrames    = SparseFXR['times_indices'].values
    TrackDurations   = SparseFXR['track_duration_sparse_indexed'].values
    DurationMask     = TrackDurations >= MinFrames
    BirthMask        = FeatureFrames == 0
    DeathMask        = FeatureFrames == (TrackDurations - 1)
    tod_mins         = BaseTimesFiveMin.hour * 60 + BaseTimesFiveMin.minute

    all_lats_raw = SparseFXR['core_meanlat'].values
    all_lons_raw = SparseFXR['core_meanlon'].values

    # ── Loop over the three modes ──────────────────────────────────────────
    for DotsIncluded in ('Birth', 'Death', 'All'):

        print(f'\n── Building density grid: {DotsIncluded} ──')

        if DotsIncluded == 'Birth':
            ModeMask = BirthMask
        elif DotsIncluded == 'Death':
            ModeMask = DeathMask
        else:
            ModeMask = np.ones(len(FeatureFrames), dtype=bool)

        # ── Pass 1: collect all points across all bins to get shared extent
        all_lats_combined = []
        all_lons_combined = []

        bin_lats = []
        bin_lons = []
        bin_counts = []

        for (t_start_str, t_end_str) in time_bins:
            t_start_mins = int(t_start_str.split(':')[0]) * 60 + int(t_start_str.split(':')[1])
            t_end_mins   = int(t_end_str.split(':')[0])   * 60 + int(t_end_str.split(':')[1])

            # handle 24:00 edge
            if t_end_mins == 1440:
                TimeMask = tod_mins >= t_start_mins
            else:
                TimeMask = (tod_mins >= t_start_mins) & (tod_mins < t_end_mins)

            ActiveMask = TimeMask & DurationMask & ModeMask

            lats = all_lats_raw[ActiveMask]
            lons = all_lons_raw[ActiveMask]
            valid = np.isfinite(lats) & np.isfinite(lons)
            lats  = lats[valid]
            lons  = lons[valid]

            bin_lats.append(lats)
            bin_lons.append(lons)
            bin_counts.append(len(lats))

            if len(lats) > 0:
                all_lats_combined.extend(lats)
                all_lons_combined.extend(lons)

            print(f'  {t_start_str}–{t_end_str} : {len(lats)} points')

        if len(all_lats_combined) == 0:
            print(f'  No points found for {DotsIncluded} — skipping.')
            continue

        all_lats_combined = np.array(all_lats_combined)
        all_lons_combined = np.array(all_lons_combined)

        # ── Shared map extent ──────────────────────────────────────────────
        lon_min = float(np.nanmin(all_lons_combined))
        lon_max = float(np.nanmax(all_lons_combined))
        lat_min = float(np.nanmin(all_lats_combined))
        lat_max = float(np.nanmax(all_lats_combined))

        # ── Shared KDE grid ────────────────────────────────────────────────
        lon_grid           = np.linspace(lon_min, lon_max, 300)
        lat_grid           = np.linspace(lat_min, lat_max, 300)
        lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
        grid_coords        = np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])
        mean_lat           = np.mean(all_lats_combined)
        km2_per_deg2       = 111.32 * 111.32 * np.cos(np.radians(mean_lat))

        # ── Pass 2: compute KDE for each bin, find global max ──────────────
        kde_grids  = []
        global_max = 0.0

        for i, (lats, lons) in enumerate(zip(bin_lats, bin_lons)):
            if len(lats) < 2:
                kde_grids.append(None)
                continue

            kde        = gaussian_kde(np.vstack([lons, lats]), bw_method=0.1)
            kde_vals   = kde(grid_coords).reshape(lon_mesh.shape)
            kde_100km2 = (kde_vals / km2_per_deg2) * len(lats) * 100.0
            kde_grids.append(kde_100km2)
            global_max = max(global_max, float(np.nanmax(kde_100km2)))

        print(f'  Global KDE max : {global_max:.3f} per 100 km²')

        # ── Colour scale ───────────────────────────────────────────────────
        level_step     = global_max / 25.0
        levels_fill    = np.arange(0, global_max + level_step, level_step)
        levels_visible = levels_fill[1:]

        # ── Build 4×2 figure ───────────────────────────────────────────────
        fig, axes = plt.subplots(
            nrows      = 2,
            ncols      = 4,
            figsize    = (24, 12),
            subplot_kw = {'projection': ccrs.PlateCarree()},
            facecolor  = 'white',
        )
        fig.subplots_adjust(right=0.88, hspace=0.15, wspace=0.05)

        for panel_i, ax in enumerate(axes.flat):

            t_start_str, t_end_str = time_bins[panel_i]
            n_pts                  = bin_counts[panel_i]
            kde_grid               = kde_grids[panel_i]

            _BuildBasemap(ax, lon_min, lon_max, lat_min, lat_max)

            if kde_grid is not None:
                # invisible contourf to anchor the shared colour scale
                ax.contourf(
                    lon_mesh, lat_mesh, kde_grid,
                    levels=levels_fill, cmap='jet', alpha=0.0,
                    transform=ccrs.PlateCarree(), zorder=24, extend='max',
                )
                # visible contourf
                ax.contourf(
                    lon_mesh, lat_mesh, kde_grid,
                    levels=levels_visible, cmap='jet', alpha=0.25,
                    transform=ccrs.PlateCarree(), zorder=24, extend='max',
                )

            # radar star
            ax.plot(RadarLon, RadarLat, marker='*', color='black',
                    markersize=8, transform=ccrs.PlateCarree(), zorder=30)
            ax.plot(RadarLon, RadarLat, marker='*', color='white',
                    markersize=4, transform=ccrs.PlateCarree(), zorder=31)

            ax.set_title(
                f'{t_start_str}–{t_end_str} UTC\n({n_pts} points)',
                fontsize=9,
            )

        # ── Shared colourbar ───────────────────────────────────────────────
        cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.70])
        sm      = plt.cm.ScalarMappable(
            cmap = 'jet',
            norm = plt.Normalize(vmin=0, vmax=global_max),
        )
        sm.set_array([])
        cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical', extend='max')
        cbar.set_label('Feature Locations per 100 km²', fontsize=11)
        tick_vals = np.linspace(0, global_max, 11)
        cbar.set_ticks(tick_vals)
        cbar.set_ticklabels([f'{v:.1f}' for v in tick_vals])
        cbar.ax.tick_params(labelsize=9)

        # ── Super title ────────────────────────────────────────────────────
        fig.suptitle(
            f'{RadarSiteName} Radar  |  {DotsIncluded} Feature Density  |  3-Hour UTC Blocks\n'
            f'{date_start.strftime("%Y-%m-%d")} to {date_end.strftime("%Y-%m-%d")}  |  '
            f'Min Duration: {MinDuration} min',
            fontsize=13,
            y=0.98,
        )

        # SavePath = SaveFolder + SaveBase + f'_{DotsIncluded}_DensityEx.png'
        # plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi=200)
        # plt.close()
        # print(f'  Saved: {SavePath}')


In [ ]:
PlotFeatureDensityGrid(
SparseFXR            = SparseFXR,
RadarIDno            = '41',
RadarSiteName        = 'Willis Island',
QualityControlOption = 2,
DateRange            = '20240201-20240229',
MinDuration          = 30,
)
